# Atlas Electoral · V22.0 — interfaz pública ordenada

**Versión estable de referencia: V21.8.** Esta entrega mantiene intactas las bases, las métricas, los indicadores, los módulos y los gráficos originales. Únicamente actualiza elementos de presentación del HTML público mediante una hoja CSS **aislada e idempotente**, sin tocar los archivos `recursos/` ni añadir salidas de figuras. Conserva el flujo reproducible `salidas/datos/` → `salidas/Atlas_Electoral_Costa_Rica_1953_2026.html`.

**Uso:** Colocar este cuaderno en el proyecto V21 y ejecutar las celdas en orden. Para las tres figuras externas del quiebre del voto utilizar el **notebook independiente**; ese generador no forma parte del Atlas público.


## 1. Entorno, rutas y presentación

La carpeta de las bases puede cambiarse en `DATA_DIR`. No se utilizan rutas de la computadora del autor ni servicios externos para generar el HTML.

In [1]:
from pathlib import Path
import sys, json, hashlib, zipfile, importlib.metadata
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML, FileLink

# Carpeta del paquete: se puede indicar una ruta absoluta si se ha movido.
import os, types
PROJECT_PATH = ""  # Si el notebook está fuera de su carpeta, indique aquí la ruta ABSOLUTA de Atlas_Electoral_V21.
# Rechazar recursos V10–V13 o una instalación parcial antes de calcular los resultados.
_REQUIRED_V14 = ("atlas_pipeline.py", "atlas_final.js", "atlas_v12.js", "atlas_v14.js", "atlas_historico_v14.py", "atlas_template.html")
if PROJECT_PATH:
    _candidates = [Path(PROJECT_PATH).expanduser()]
else:
    _cwd = Path.cwd()
    _candidates = [
        _cwd, _cwd / "Atlas_Electoral_V21", _cwd / "Atlas_Electoral_V20",
        _cwd.parent / "Atlas_Electoral_V21", _cwd.parent / "Atlas_Electoral_V20",
        Path.home() / "Documents" / "GitHub" / "Atlas_Electoral_V21",
        Path.home() / "Downloads" / "Atlas_Electoral_V21",
    ]
PROJECT_DIR = next((c.resolve() for c in _candidates
                    if c.is_dir() and all((c / "recursos" / name).is_file() for name in _REQUIRED_V14)), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "ATLAS V21: no se encontró una carpeta COMPLETA con recursos/atlas_v12.js, "
        "recursos/atlas_final.js, recursos/atlas_historico_v14.py, recursos/atlas_template.html y recursos/atlas_pipeline.py. "
        "Esto sucede si se mezcló el notebook V10–V13 con los recursos V13 o si el ZIP "
        "no se extrajo íntegramente. Abra este notebook "
        "dentro de Atlas_Electoral_V21, reinicie el kernel y ejecute todo. "
        "Carpeta de Jupyter detectada: " + str(Path.cwd())
    )
print("Atlas V21: recursos cargados desde", PROJECT_DIR)
os.chdir(PROJECT_DIR)
DATA_DIR = PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR / "salidas"
for folder in (OUTPUT_DIR, OUTPUT_DIR / "datos"):
    folder.mkdir(parents=True, exist_ok=True)

# Cargar el código de ESTE paquete, incluso si Jupyter tenía una versión anterior.
RESOURCE_DIR = PROJECT_DIR / "recursos"
sys.path.insert(0, str(RESOURCE_DIR))
for module_name in ("atlas_enhancements", "atlas_geography", "atlas_outcomes", "atlas_pipeline"):
    module_path = RESOURCE_DIR / (module_name + ".py")
    module = types.ModuleType(module_name)
    module.__file__ = str(module_path)
    sys.modules[module_name] = module
    exec(compile(module_path.read_text(encoding="utf-8"), str(module_path), "exec"), module.__dict__)
from atlas_pipeline import build_dataset, render_app, FILES, YEARS, PROVINCES

palette_path = PROJECT_DIR / "recursos" / "colores_partidos.json"
PARTY_COLORS = {r["partido"]: r["color"] for r in json.loads(palette_path.read_text(encoding="utf-8")) if r.get("color")}
PROVINCE_COLORS = {"San José":"#0072B2", "Alajuela":"#D55E00", "Cartago":"#009E73", "Heredia":"#8B55A2", "Guanacaste":"#B18A00", "Puntarenas":"#CB5D92", "Limón":"#3A8F9A"}
SEX_COLORS = {"Mujer":"#933B8F", "Hombre":"#167D8D"}
COLORS = ["#007f7a", "#3276c2", "#c29049", "#a55579", "#6266a4", "#5b8f42"]
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = COLORS
# El destino primario del notebook es el HTML. Evitar renderizar repetidamente
# decenas de Plotly dentro del cuaderno reduce memoria y tiempo de ejecución.
# Para previsualizar las figuras de EDA dentro de Jupyter, cambie a True.
NOTEBOOK_INLINE_PLOTS = False
_plotly_loaded = False

def show(fig, title=None):
    """Salida Plotly sin CDN, con estilo y descarga PNG."""
    global _plotly_loaded
    if not NOTEBOOK_INLINE_PLOTS: return
    for trace in fig.data:
        color = {**PARTY_COLORS, **PROVINCE_COLORS, **SEX_COLORS,
                 "Legislativa":"#126CB5", "Presidencial":"#D05A22"}.get(trace.name)
        if color:
            if hasattr(trace, "marker"): trace.marker.color = color
            if hasattr(trace, "line"): trace.line.color = color
        if getattr(trace, "type", None) == "scatter" and trace.name == "Presidencial":
            trace.line.dash = "dash"
            trace.marker.symbol = "diamond"
    fig.update_layout(font=dict(family="Arial", color="#172c3c"),
                      title=title or fig.layout.title.text,
                      margin=dict(l=65, r=25, t=60, b=60), height=460,
                      legend=dict(orientation="h", y=-0.22))
    display(HTML(fig.to_html(full_html=False, include_plotlyjs=not _plotly_loaded,
        config={"displaylogo": False, "responsive": True,
                "toImageButtonOptions": {"format": "png", "scale": 3}})))
    _plotly_loaded = True

print("Entorno:", {p: importlib.metadata.version(p) for p in ("pandas", "numpy", "openpyxl", "plotly")})


Atlas V21: recursos cargados desde /Users/juani/Documents/GitHub/atlas-electoral-costa-rica
Entorno: {'pandas': '2.3.3', 'numpy': '2.0.2', 'openpyxl': '3.1.5', 'plotly': '6.9.0'}


In [2]:
# Nombres de las siete bases: admitir nombres limpios y sufijos históricos sin tocar los archivos.
from pathlib import Path
import re, unicodedata

def _nombre_atlas(nombre):
    texto = ''.join(ch for ch in unicodedata.normalize('NFD', str(nombre).lower()) if unicodedata.category(ch) != 'Mn')
    return re.sub(r'[^a-z0-9]+', '_', texto).strip('_')

def resolver_bases_atlas(carpeta, archivos):
    carpeta = Path(carpeta)
    patrones = {
        'leg': [r'base.*electoral.*diput', r'resultados?.*legislat'],
        'pres': [r'base.*electoral.*presiden', r'resultados?.*presiden'],
        'cand': [r'candidat.*diput', r'base.*candidat.*legislat'],
        'formula': [r'candidat.*presiden', r'formula.*presiden'],
        'electas': [r'diputad.*elect', r'diputacion.*elect', r'diputaciones.*elect'],
        'diseno': [r'disen.*papelet', r'papelet.*diput'],
        'cancelados': [r'partid.*cancelad'],
    }
    disponibles = sorted(p for p in carpeta.iterdir() if p.is_file() and p.suffix.lower() in ('.xlsx', '.xlsm'))
    candidatos = {p: _nombre_atlas(p.stem) for p in disponibles}
    seleccion = {}
    for rol, anteriores in archivos.items():
        anterior = carpeta / anteriores
        if anterior.is_file():
            seleccion[rol] = anterior.name
            continue
        opciones = [p for p, nombre in candidatos.items() if any(re.search(pat, nombre) for pat in patrones[rol])]
        # Excluir otras familias cuando se comparten palabras.
        if rol == 'leg': opciones = [p for p in opciones if 'candidat' not in candidatos[p] and 'electas' not in candidatos[p]]
        if rol == 'pres': opciones = [p for p in opciones if 'candidat' not in candidatos[p]]
        if rol == 'cand': opciones = [p for p in opciones if 'presiden' not in candidatos[p] and 'electas' not in candidatos[p]]
        if rol == 'electas': opciones = [p for p in opciones if 'candidat' not in candidatos[p]]
        if rol == 'diseno': opciones = [p for p in opciones if 'presiden' not in candidatos[p]]
        if len(opciones) == 1:
            seleccion[rol] = opciones[0].name
        elif not opciones:
            raise FileNotFoundError(f'No se encontró la base «{rol}» en {carpeta}. Archivos Excel encontrados: {[x.name for x in disponibles]}')
        else:
            raise RuntimeError(f'Hay varias bases compatibles con «{rol}»: {[x.name for x in opciones]}. Dejá una sola versión para evitar usar datos equivocados.')
    assert len(set(seleccion.values())) == len(archivos), 'Se intentó asignar el mismo Excel a dos bases distintas.'
    archivos.update(seleccion)  # Mismo diccionario importado desde atlas_pipeline.
    print('Siete fuentes localizadas (sin cambiar el nombre de tus Excel):')
    for rol, nombre in archivos.items(): print(f'  {rol:12} → {nombre}')

resolver_bases_atlas(DATA_DIR, FILES)


Siete fuentes localizadas (sin cambiar el nombre de tus Excel):
  leg          → Base_Electoral_Diputaciones_1953-2026.xlsx
  pres         → Base_Electoral_Presidencial_1953-2026.xlsx
  cand         → Base_Final_Candidaturas_Diputaciones_1953-2026.xlsx
  formula      → Base_Final_Candidaturas_Presidenciales_1953-2026.xlsx
  electas      → Base_Final_Diputaciones_Electas_1953-2026.xlsx
  diseno       → Base_Final_Diseno_Papeletas_Diputaciones_1953-2026.xlsx
  cancelados   → Base_partidos_cancelados.xlsx


In [3]:
# V5 · Parche compatible e idempotente para el Atlas Electoral
# Sustituye los parches V2/V3/V4 basados en coincidencias literales.
# Mantiene el diseño y los datos, mejora exportación PNG/SVG de personas,
# marca hitos del TSE sobre la serie de género y añade dos tasas junto a los flujos.
from pathlib import Path
import shutil

asset = PROJECT_DIR / 'recursos' / 'atlas_final.js'
if not asset.is_file():
    raise FileNotFoundError(f'No existe {asset}. Ajuste PROJECT_PATH en Entorno: el notebook debe ejecutarse desde la raíz del Atlas.')
raw = asset.read_text(encoding='utf-8')
required = ['views.outcomes=', 'Composición por sexo a través del tiempo', 'transiciones_resultados']
missing = [marker for marker in required if marker not in raw]
if missing:
    raise RuntimeError('El archivo atlas_final.js no parece pertenecer a una versión compatible del Atlas Electoral (V2.0): '+', '.join(missing)+'; encontrado en '+str(asset))
PATCH_MARKER = 'ATLAS_V5_COMPATIBLE_PATCH'
if PATCH_MARKER not in raw:
    extension = 'async function atlasV5ExportNode(id, format=\'png\', filename=\'Atlas_Exportacion\'){\n // ATLAS_VECTOR_EXPORT_V5: elementos SVG nativos, sin HTML/CSS incrustado.\n try {\n  const root=document.getElementById(id);\n  if(!root)throw new Error(\'No se encontró la sección de personas seleccionada.\');\n  const cardNodes=Array.from(root.querySelectorAll(\'.person-result\'));\n  if(!cardNodes.length)throw new Error(\'Esta selección no contiene personas que se puedan exportar.\');\n  const compact=s=>String(s??\'\').replace(/\\s+/g,\' \').trim();\n  const readCard=el=>{\n   const name=compact(el.querySelector(\'.person-link\')?.textContent||el.querySelector(\':scope > span\')?.textContent||\'Persona sin nombre\');\n   const badges=Array.from(el.querySelectorAll(\'.outcome-badge\')).map(n=>compact(n.textContent)).filter(Boolean);\n   const pairs=Array.from(el.querySelectorAll(\'.career-pair\')).map(n=>compact(n.textContent)).filter(Boolean);\n   const extras=Array.from(el.querySelectorAll(\'.mini,.route\')).filter(n=>!n.closest(\'.career-pair\')).map(n=>compact(n.textContent)).filter(Boolean);\n   return {name,badges,details:[...pairs,...extras]};\n  };\n  const people=cardNodes.map(readCard);\n  const heading=compact(root.querySelector(\'h3\')?.textContent||\'Personas de la selección\');\n  const scope=compact(root.querySelector(\'.card-head .mini\')?.textContent||\'\');\n  const methodology=compact(root.querySelector(\':scope > .chart-note\')?.textContent||\'\');\n  const cols=people.length>36?3:people.length>5?2:1;\n  const width=cols===3?2100:cols===2?1560:1120;\n  const margin=55, gap=22, cardWidth=(width-2*margin-gap*(cols-1))/cols;\n  const wrap=(value,max)=>{\n   const words=compact(value).split(\' \').filter(Boolean),out=[];let line=\'\';\n   for(const word of words){\n    if(word.length>max){if(line){out.push(line);line=\'\';}for(let i=0;i<word.length;i+=max)out.push(word.slice(i,i+max));continue;}\n    if((line?line.length+1:0)+word.length>max){out.push(line);line=word;}else line+=(line?\' \':\'\')+word;\n   }\n   if(line)out.push(line);\n   return out.length?out:[\'\'];\n  };\n  const maxTitle=cols===3?43:cols===2?59:100;\n  const maxDetail=cols===3?48:cols===2?65:105;\n  const prepared=people.map(p=>({\n   name:wrap(p.name,maxTitle),\n   badges:p.badges.flatMap(s=>wrap(s,maxDetail)),\n   details:p.details.flatMap(s=>wrap(s,maxDetail)),\n  }));\n  const cardHeight=p=>24+p.name.length*29+(p.badges.length?8+p.badges.length*23:0)+(p.details.length?9+p.details.length*24:0)+22;\n  const headScope=wrap(scope,Math.max(55,Math.floor((width-2*margin)/11)));\n  const headHeight=184+Math.max(0,headScope.length-1)*25;\n  const placement=[];let nextY=headHeight;\n  for(let i=0;i<prepared.length;i+=cols){\n   const slice=prepared.slice(i,i+cols);\n   const rh=Math.max(...slice.map(cardHeight));\n   slice.forEach((p,j)=>placement.push({person:p,x:margin+j*(cardWidth+gap),y:nextY,height:rh}));\n   nextY+=rh+gap;\n  }\n  const noteLines=methodology?wrap(methodology,Math.floor((width-2*margin)/10)):[];\n  const totalHeight=Math.ceil(nextY+46+noteLines.length*24+60);\n  const ns=\'http://www.w3.org/2000/svg\';\n  const svg=document.createElementNS(ns,\'svg\');\n  svg.setAttribute(\'xmlns\',ns);svg.setAttribute(\'width\',String(width));\n  svg.setAttribute(\'height\',String(totalHeight));svg.setAttribute(\'viewBox\',`0 0 ${width} ${totalHeight}`);\n  svg.setAttribute(\'role\',\'img\');svg.setAttribute(\'aria-label\',heading);\n  const draw=(tag,attrs,content,parent=svg)=>{\n   const e=document.createElementNS(ns,tag);\n   for(const [key,value] of Object.entries(attrs||{}))e.setAttribute(key,String(value));\n   if(content!==null&&content!==undefined)e.textContent=String(content);\n   parent.appendChild(e);return e;\n  };\n  const tx=(x,y,value,size=18,color=\'#274251\',weight=400)=>draw(\'text\',{x,y,fill:color,\'font-family\':\'Arial, Helvetica, sans-serif\',\'font-size\':size,\'font-weight\':weight},value);\n  draw(\'rect\',{x:0,y:0,width,height:totalHeight,fill:\'#f5f8fa\'});\n  draw(\'rect\',{x:0,y:0,width,height:10,fill:\'#137f7a\'});\n  tx(margin,51,\'ATLAS ELECTORAL DE COSTA RICA · 1953–2026\',20,\'#167a78\',700);\n  tx(margin,94,heading,29,\'#173143\',700);\n  headScope.forEach((s,i)=>tx(margin,130+i*25,s,17,\'#466474\',400));\n  tx(margin,headHeight-20,`PERSONAS EN ESTA PÁGINA: ${people.length} · Filtros indicados en el encabezado`,17,\'#3f6170\',700);\n  draw(\'line\',{x1:margin,y1:headHeight-7,x2:width-margin,y2:headHeight-7,stroke:\'#bfd4d6\',\'stroke-width\':2});\n  placement.forEach(({person:p,x,y,height})=>{\n   draw(\'rect\',{x,y,width:cardWidth,height,rx:11,fill:\'#ffffff\',stroke:\'#cfdee5\',\'stroke-width\':1.5});\n   draw(\'rect\',{x,y,width:7,height,rx:3,fill:\'#137f7a\'});\n   let baseline=y+36;\n   p.name.forEach(line=>{tx(x+24,baseline,line,21,\'#153a4c\',700);baseline+=29;});\n   if(p.badges.length){baseline+=5;p.badges.forEach(line=>{tx(x+24,baseline,line,16,\'#0c746b\',700);baseline+=23;});}\n   if(p.details.length){baseline+=6;p.details.forEach(line=>{tx(x+24,baseline,line,16,\'#415c69\',400);baseline+=24;});}\n  });\n  const footY=nextY+17;\n  draw(\'line\',{x1:margin,y1:footY-16,x2:width-margin,y2:footY-16,stroke:\'#d4e1e7\',\'stroke-width\':1.5});\n  noteLines.forEach((line,i)=>tx(margin,footY+i*24,line,15,\'#526874\'));\n  tx(margin,totalHeight-27,\'Fuente: Atlas electoral de Costa Rica · Consulta filtrada · Resultados descriptivos del corpus\',15,\'#536a77\');\n  const serialized=new XMLSerializer().serializeToString(svg);\n  const vectorBlob=new Blob([serialized],{type:\'image/svg+xml;charset=utf-8\'});\n  const safeName=(filename||\'Atlas_Personas\').replace(/[^\\p{L}\\p{N}_-]+/gu,\'_\');\n  const saveBlob=(blob,ext)=>{\n   const u=URL.createObjectURL(blob),a=document.createElement(\'a\');\n   a.href=u;a.download=`${safeName}.${ext}`;document.body.appendChild(a);a.click();a.remove();\n   setTimeout(()=>URL.revokeObjectURL(u),2000);\n  };\n  if(format===\'svg\'){saveBlob(vectorBlob,\'svg\');return;}\n  if(format!==\'png\')throw new Error(\'Formato de exportación desconocido.\');\n  const imageUrl=URL.createObjectURL(vectorBlob);\n  try {\n   const bitmap=new Image();\n   await new Promise((resolve,reject)=>{\n    bitmap.onload=resolve;\n    bitmap.onerror=()=>reject(new Error(\'El navegador no pudo cargar la representación vectorial de los nombres.\'));\n    bitmap.src=imageUrl;\n   });\n   const factor=Math.max(.6,Math.min(2,15000/totalHeight,Math.sqrt(45000000/(width*totalHeight))));\n   const canvas=document.createElement(\'canvas\');\n   canvas.width=Math.max(1,Math.round(width*factor));\n   canvas.height=Math.max(1,Math.round(totalHeight*factor));\n   const ctx=canvas.getContext(\'2d\');\n   if(!ctx)throw new Error(\'No se pudo inicializar el lienzo de imagen.\');\n   ctx.fillStyle=\'#ffffff\';ctx.fillRect(0,0,canvas.width,canvas.height);\n   ctx.drawImage(bitmap,0,0,canvas.width,canvas.height);\n   const raster=await new Promise((resolve,reject)=>canvas.toBlob(b=>b?resolve(b):reject(new Error(\'El navegador no pudo convertir el gráfico en PNG.\')),\'image/png\'));\n   saveBlob(raster,\'png\');\n  } finally {URL.revokeObjectURL(imageUrl);}\n }catch(error){console.error(\'Atlas · fallo de exportación:\',error);alert(\'No se pudo exportar la imagen: \'+(error?.message||error));}\n}\n\n/* ATLAS_V5_COMPATIBLE_PATCH\n * Ampliación no destructiva de la última capa de la interfaz.\n * No depende del texto exacto de atlas_final.js; se puede aplicar a clones V2.0\n * o a copias parcialmente parcheadas. No sustituye la aplicación existente.\n */\n(function atlasV5Enhance(){\n  const stamp=\'ATLAS_V5_COMPATIBLE_PATCH\';\n  if (typeof views===\'undefined\' || typeof views.outcomes!==\'function\' || typeof plot!==\'function\') {\n    throw new Error(\'El paquete no expone las funciones esperadas del Atlas. Compruebe que recursos/atlas_final.js corresponda a la versión 2.0.\');\n  }\n  const milestones=[\n    {year:1998,label:\'Cuota 40 %\',detail:\'Ley 7653 (1996): cuota mínima de candidaturas\'},\n    {year:2002,label:\'Puestos elegibles\',detail:\'TSE 1863-1999: aplicación en posiciones elegibles\'},\n    {year:2014,label:\'Paridad vertical\',detail:\'Código Electoral de 2009: paridad y alternancia\'},\n    {year:2018,label:\'Paridad horizontal\',detail:\'TSE 3603-E8-2016: encabezamientos provinciales\'}\n  ];\n  // Amplía opciones del gráfico EXISTENTE sin modificar barras, datos ni resultados.\n  const previousPlot=plot;\n  plot=function(titleText,traces,rows,opts={}){\n    if(titleText===\'Composición por sexo a través del tiempo\'){\n      const years=new Set(traces.flatMap(t=>Array.isArray(t.x)?t.x.map(Number):[]));\n      const active=milestones.filter(m=>years.has(m.year));\n      const configured=Array.isArray(opts.layout?.shapes)&&opts.layout.shapes.some(s=>active.some(m=>Number(s.x0)===m.year && s.type===\'line\'));\n      if(active.length && !configured){\n        const shapes=active.map(m=>({type:\'line\',xref:\'x\',yref:\'paper\',x0:m.year,x1:m.year,y0:0,y1:1,line:{color:\'#b28744\',width:1.75,dash:\'dot\'}}));\n        const annotations=active.map((m,i)=>({xref:\'x\',yref:\'paper\',x:m.year,y:i%2?1.18:1.09,text:`${m.year} · ${m.label}`,showarrow:false,textangle:-30,font:{size:10,color:\'#755c32\'},bgcolor:\'rgba(255,250,241,.97)\',bordercolor:\'rgba(178,135,68,.36)\',borderpad:3}));\n        opts={...opts,height:Math.max(opts.height||420,515),layout:{...(opts.layout||{}),shapes:[...(opts.layout?.shapes||[]),...shapes],annotations:[...(opts.layout?.annotations||[]),...annotations],margin:{...(opts.layout?.margin||{}),t:145}},note:[opts.note,\'Líneas doradas: hitos en la elección de aplicación (1998: cuota 40 %; 2002: puestos elegibles; 2014: paridad vertical; 2018: paridad horizontal). Normas: Ley 7653/1996; TSE 1863-1999; Código Electoral 2009; TSE 3603-E8-2016. La coincidencia temporal no demuestra causalidad.\'].filter(Boolean).join(\' \')};\n      }\n    }\n    if(titleText===\'Evolución del indicador territorial\' && (local.rmetric||\'indice_representacion\')===\'indice_representacion\'){\n      const hasLine=opts.layout?.shapes?.some(s=>s.yref===\'y\'&&Number(s.y0)===1&&Number(s.y1)===1);\n      if(!hasLine){\n        opts={...opts,layout:{...(opts.layout||{}),shapes:[...(opts.layout?.shapes||[]),{type:\'line\',xref:\'paper\',yref:\'y\',x0:0,x1:1,y0:1,y1:1,line:{color:\'#b45353\',width:2,dash:\'dot\'}}]},note:[opts.note,\'La línea roja y = 1 expresa igualdad descriptiva entre peso en escaños y peso en padrón, no la regla constitucional de distribución por población.\'].filter(Boolean).join(\' \')};\n      }\n    }\n    return previousPlot(titleText,traces,rows,opts);\n  };\n\n  // Los controles se montan sobre el bloque VISIBLE, cualquiera que sea su\n  // implementación interna. Esto evita modificar versiones antiguas de views.outcomes.\n  function ensurePersonExports(){\n    const roots=Array.from(document.querySelectorAll(\'#content .card\'));\n    const wanted=new Map([\n      [\'Personas del flujo seleccionado\',\'Atlas_ExitoElectoral_FlujoSeleccionado\'],\n      [\'Quiénes forman este puente de carrera\',\'Atlas_ExitoElectoral_PuenteCarrera\'],\n      [\'Personas del antecedente seleccionado\',\'Atlas_ExitoElectoral_AntecedenteSeleccionado\'],\n      [\'Personas en el antecedente seleccionado\',\'Atlas_ExitoElectoral_AntecedenteSeleccionado\']\n    ]);\n    roots.forEach(root=>{\n      const heading=root.querySelector(\'h3\');\n      if(!heading)return;\n      const name=heading.textContent.trim();\n      if(!wanted.has(name) || root.dataset.atlasV5Export===\'yes\')return;\n      root.dataset.atlasV5Export=\'yes\';\n      if(!root.id)root.id=\'atlas_v5_people_\'+Array.from(root.parentNode.children).indexOf(root)+\'_\'+Math.floor(Math.random()*100000);\n      // Reemplazar solamente los controles exportadores del bloque de personas.\n      // Los botones de las figuras Plotly y de las tablas no se modifican.\n      let head=heading.closest(\'.card-head\');\n      if(!head){head=document.createElement(\'div\');head.className=\'card-head\';heading.parentNode.insertBefore(head,heading);head.appendChild(heading);}\n      head.querySelectorAll(\'.exports\').forEach(el=>el.remove());\n      const tools=document.createElement(\'div\');tools.className=\'exports\';\n      for(const kind of [\'PNG\',\'SVG\']){\n        const b=document.createElement(\'button\');b.type=\'button\';b.textContent=kind;\n        b.title=`Descargar los nombres y trayectorias de esta página como ${kind}`;\n        b.addEventListener(\'click\',()=>atlasV5ExportNode(root.id,kind.toLowerCase(),wanted.get(name)));\n        tools.appendChild(b);\n      }\n      head.appendChild(tools);\n      const mini=root.querySelector(\'.card-head .mini\');\n      if(!mini){\n        const scope=document.createElement(\'span\');scope.className=\'mini\';\n        scope.textContent=`${state.from}–${state.to} · ${state.province} · cargo: ${local.ocargo||\'Todos\'} · partido: ${local.oparty||\'Todos\'} · sexo: ${local.osex||\'Todos\'}`;\n        heading.insertAdjacentElement(\'afterend\',scope);\n      }\n    });\n  }\n\n  function addTransitionRates(){\n    const titles=Array.from(document.querySelectorAll(\'#content .card h3\')).map(x=>x.textContent.trim());\n    if(titles.includes(\'Tasa de elección posterior según continuidad de partido\'))return;\n    const rows=typeof atlasFilteredOutcomes===\'function\'?atlasFilteredOutcomes():[];\n    const ids=new Set(rows.map(r=>r.id_persona));\n    let transitions=(D.transiciones_resultados||[]).filter(r=>r.anio>=state.from&&r.anio<=state.to&&ids.has(r.id_persona));\n    if(state.province!==\'Nacional\')transitions=transitions.filter(r=>r.provincia_origen===state.province||r.provincia_destino===state.province);\n    add(\'<div class="extension-heading">LECTURA PORCENTUAL DE LOS FLUJOS · PARTIDO Y PROVINCIA</div>\');\n    add(\'<div class="clarity-box"><b>Complemento, no sustitución.</b> Los diagramas de flujo muestran personas y transiciones. Las dos tasas siguientes se calculan con el número de transiciones como denominador; una persona puede aportar varias. Son asociaciones descriptivas, no pruebas causales.</div>\');\n    const context=`${state.from}–${state.to} · ${state.province} · Cargo: ${local.ocargo||\'Todos\'} · Sexo: ${local.osex||\'Todos\'} · Partido: ${labelText(local.oparty||\'Todos\')}`;\n    const one=(which,title)=>{\n      const a=which===\'provincia\'?transitions.filter(r=>r.provincia_origen!==\'Nacional\'&&r.provincia_destino!==\'Nacional\'):transitions;\n      const key=which===\'provincia\'?\'cambia_provincia\':\'cambia_partido\';\n      const groups=[[`Mantiene ${which}`,false],[`Cambia ${which}`,true]];\n      const stats=groups.map(([grupo,change])=>{\n        const g=a.filter(r=>Boolean(r[key])===change),won=g.filter(r=>Boolean(r.electa_destino)).length;\n        return {grupo,transiciones:g.length,electas_despues:won,no_electas_despues:g.length-won,tasa:g.length?100*won/g.length:null,personas:uniq(g.map(r=>r.id_persona)).length};\n      });\n      const validRows=stats.filter(x=>x.transiciones>0);\n      if(!validRows.length){note(`No hay transiciones para ${title} con los filtros actuales.`);return;}\n      plot(title,[{type:\'bar\',orientation:\'h\',x:validRows.map(r=>r.tasa),y:validRows.map(r=>r.grupo),text:validRows.map(r=>`${num(r.tasa,1)} % · ${r.electas_despues}/${r.transiciones}`),textposition:\'outside\',marker:{color:validRows.map(r=>r.grupo.startsWith(\'Mantiene\')?\'#168278\':\'#637cac\')},customdata:validRows.map(r=>[r.electas_despues,r.transiciones,r.personas]),hovertemplate:\'%{y}<br>Electas: %{customdata[0]}/%{customdata[1]} transiciones · %{customdata[2]} personas<br>%{x:.2f} %<extra></extra>\'}],validRows,{height:310,xTitle:\'Transiciones con elección posterior / transiciones del grupo (%)\',subtitle:context,note:\'N = transiciones registradas entre elecciones distintas. El gráfico complementa el Sankey; mantener el partido o la provincia no implica causalidad ni una recomendación individual.\'});\n      table(stats,[\'grupo\',\'transiciones\',\'electas_despues\',\'no_electas_despues\',\'tasa\',\'personas\'],`Auditoría · ${title}`);\n    };\n    one(\'partido\',\'Tasa de elección posterior según continuidad de partido\');\n    one(\'provincia\',\'Tasa de elección posterior según continuidad de provincia\');\n  }\n  const previousOutcomes=views.outcomes;\n  views.outcomes=function(...args){\n    const result=previousOutcomes.apply(this,args);\n    ensurePersonExports();\n    addTransitionRates();\n    return result;\n  };\n})();\n'
    asset.write_text(raw+'\n'+extension+'\n', encoding='utf-8')
    print('V5 aplicada en', asset)
else:
    print('V5 ya estaba aplicada: no se ha duplicado ningún gráfico ni control.')
print('Versión compatible · El HTML se generará al ejecutar la celda 20 del notebook.')


V5 ya estaba aplicada: no se ha duplicado ningún gráfico ni control.
Versión compatible · El HTML se generará al ejecutar la celda 20 del notebook.


In [4]:
# V6 · Claridad visual para el artículo
# Mejora legibilidad del módulo de género y las tasas de continuidad
# sin reemplazar la lógica del Atlas. Es idempotente: si ya fue aplicado,
# no duplica cambios.
from pathlib import Path
import shutil

asset = PROJECT_DIR / 'recursos' / 'atlas_final.js'
if not asset.is_file():
    raise FileNotFoundError(f'No existe {asset}. Ajuste PROJECT_PATH en Entorno: el notebook debe ejecutarse desde la raíz del Atlas.')
raw = asset.read_text(encoding='utf-8')
required = ['views.genero=', 'views.outcomes=', 'Composición por sexo a través del tiempo']
missing = [marker for marker in required if marker not in raw]
if missing:
    raise RuntimeError('El archivo atlas_final.js no parece pertenecer a una versión compatible del Atlas: '+', '.join(missing)+'; encontrado en '+str(asset))
PATCH_MARKER = 'ATLAS_V6_READABILITY_PATCH'
if PATCH_MARKER not in raw:
    extension = r'''
/* ATLAS_V6_READABILITY_PATCH
 * Mejoras de legibilidad para el artículo: género y éxito electoral.
 */
(function atlasV6ReadabilityPatch(){
  const stamp='ATLAS_V6_READABILITY_PATCH';
  if(typeof views==='undefined' || typeof plot!=='function') return;
  const milestones=[
    {year:1998,label:'1998 · Cuota 40 %',detail:'Ley 7653 (1996): cuota mínima de candidaturas'},
    {year:2002,label:'2002 · Puestos elegibles',detail:'TSE 1863-1999: aplicación en posiciones elegibles'},
    {year:2014,label:'2014 · Paridad vertical',detail:'Código Electoral de 2009: paridad y alternancia'},
    {year:2018,label:'2018 · Paridad horizontal',detail:'TSE 3603-E8-2016: encabezamientos provinciales'}
  ];
  const previousPlot=plot;
  plot=function(titleText,traces,rows,opts={}){
    let title=titleText;
    let nextTraces=Array.isArray(traces)?traces.map(t=>({...t})):traces;
    let nextOpts={...(opts||{})};
    const sexTitleMatch=String(titleText||'').startsWith('Composición por sexo');
    if(sexTitleMatch){
      const pop=(typeof local!=='undefined' && local.sexpop)?local.sexpop:'';
      const party=(typeof local!=='undefined' && local.sexparty)?labelText(local.sexparty):'Todos';
      if(titleText==='Composición por sexo a través del tiempo' && pop) title=`Composición por sexo a través del tiempo · ${pop}`;
      if(Array.isArray(nextTraces)){
        nextTraces=nextTraces.map(t=>({
          ...t,
          textfont:{size:10,color:'#ffffff'},
          insidetextanchor:'middle',
          cliponaxis:false
        }));
      }
      if(String(title).includes('a través del tiempo')){
        const years=new Set((Array.isArray(nextTraces)?nextTraces:[]).flatMap(t=>Array.isArray(t.x)?t.x.map(Number):[]));
        const active=milestones.filter(m=>years.has(m.year));
        const shapes=active.map(m=>({type:'line',xref:'x',yref:'paper',x0:m.year,x1:m.year,y0:0,y1:1,line:{color:'#b28744',width:1.75,dash:'dot'}}));
        const yRows=[1.08,1.14,1.08,1.14];
        const annotations=active.map((m,i)=>({xref:'x',yref:'paper',x:m.year,y:yRows[i%yRows.length],text:m.label,showarrow:false,textangle:0,xanchor:'center',font:{size:10,color:'#755c32'},bgcolor:'rgba(255,250,241,.98)',bordercolor:'rgba(178,135,68,.38)',borderpad:3}));
        nextOpts={
          ...nextOpts,
          height:Math.max(nextOpts.height||420,560),
          subtitle:[nextOpts.subtitle, `${state.from}–${state.to} · ${state.province} · Población: ${pop||'No especificada'} · Partido: ${party}`].filter(Boolean).join(' · '),
          layout:{
            ...(nextOpts.layout||{}),
            barmode:'stack',
            yaxis:{...((nextOpts.layout||{}).yaxis||{}), range:[0,100]},
            shapes:[...(((nextOpts.layout||{}).shapes)||[]), ...shapes],
            annotations:[...(((nextOpts.layout||{}).annotations)||[]), ...annotations],
            margin:{...(((nextOpts.layout||{}).margin)||{}), t:170}
          },
          note:[nextOpts.note,'Líneas doradas: hitos de la normativa electoral sobre participación de mujeres. Su coincidencia con cambios en la serie es descriptiva; no demuestra causalidad por sí sola.'].filter(Boolean).join(' ')
        };
      } else {
        nextOpts={...nextOpts, subtitle:[nextOpts.subtitle, `${state.from}–${state.to} · ${state.province} · Población: ${pop||'No especificada'} · Partido: ${party}`].filter(Boolean).join(' · ')};
      }
    }
    if(titleText==='Tasa de elección posterior según continuidad de partido'){
      title='Tasa de elección posterior según continuidad de partido · candidaturas posteriores';
      nextTraces=(Array.isArray(nextTraces)?nextTraces:[nextTraces]).map(t=>({...t,textposition:'outside',cliponaxis:false}));
      nextOpts={...nextOpts,xTitle:'Porcentaje de transiciones del grupo que terminan en una candidatura electa (%)',subtitle:[nextOpts.subtitle,'Lectura: de todas las transiciones observadas en cada grupo, ¿qué porcentaje terminó en una candidatura electa posterior?'].filter(Boolean).join(' · '),note:[nextOpts.note,'N del rótulo = candidaturas electas posteriores / transiciones observadas dentro del grupo.'].filter(Boolean).join(' '),height:340};
    }
    if(titleText==='Tasa de elección posterior según continuidad de provincia'){
      title='Tasa de elección posterior según continuidad de provincia · candidaturas posteriores';
      nextTraces=(Array.isArray(nextTraces)?nextTraces:[nextTraces]).map(t=>({...t,textposition:'outside',cliponaxis:false}));
      nextOpts={...nextOpts,xTitle:'Porcentaje de transiciones del grupo que terminan en una candidatura electa (%)',subtitle:[nextOpts.subtitle,'Lectura: de todas las transiciones observadas en cada grupo, ¿qué porcentaje terminó en una candidatura electa posterior?'].filter(Boolean).join(' · '),note:[nextOpts.note,'N del rótulo = candidaturas electas posteriores / transiciones observadas dentro del grupo.'].filter(Boolean).join(' '),height:340};
    }
    return previousPlot(title,nextTraces,rows,nextOpts);
  };

  const previousGenero=views.genero;
  views.genero=function(...args){
    const result=previousGenero.apply(this,args);
    const pop=local.sexpop||'Diputación propietaria';
    let base=[], outcomeApplicable=true;
    if(pop==='Diputación propietaria')base=period(D.candidaturas).filter(r=>r.tipo==='Propietaria');
    else if(pop==='Suplencia legislativa'){base=period(D.candidaturas).filter(r=>r.tipo==='Suplencia'); outcomeApplicable=false;}
    else if(pop==='Diputación electa'){base=period(D.electas); outcomeApplicable=false;}
    else base=period(D.formulas,false).filter(r=>r.cargo===pop);
    const parties=['Todos',...uniq(base.map(r=>r.partido).filter(Boolean)).sort()];
    const party=(parties.includes(local.sexparty)?local.sexparty:'Todos');
    base=base.filter(r=>party==='Todos'||r.partido===party);
    const makeComp=(rows,title,noteText)=>{
      const ys=yearsVisible();
      const comp=ys.flatMap(anio=>{const g=rows.filter(r=>r.anio===anio), den=g.filter(r=>['Mujer','Hombre'].includes(r.sexo)).length; return ['Mujer','Hombre'].map(sexo=>({anio,sexo,registros:g.filter(r=>r.sexo===sexo).length,total_con_sexo:den,porcentaje:pct(g.filter(r=>r.sexo===sexo).length,den)}));});
      const visible=ys.filter(y=>comp.some(r=>r.anio===y&&r.total_con_sexo>0));
      if(!visible.length){note(`No hay registros suficientes para “${title}” con los filtros actuales.`); return;}
      plot(title,['Mujer','Hombre'].map(sexo=>({type:'bar',name:sexo,x:visible,y:visible.map(y=>comp.find(r=>r.anio===y&&r.sexo===sexo)?.porcentaje??0),customdata:visible.map(y=>comp.find(r=>r.anio===y&&r.sexo===sexo)?.registros??0),marker:{color:sexColors[sexo]},text:visible.map(y=>{const z=comp.find(r=>r.anio===y&&r.sexo===sexo); return `${num(z?.porcentaje??0,1)}% · N=${z?.registros??0}`;}),hovertemplate:'%{x}<br>'+sexo+': %{customdata} registros · %{y:.1f}%<extra></extra>'})),comp,{height:420,yTitle:'Composición (%)',layout:{barmode:'stack',yaxis:{range:[0,100]}},note:noteText});
    };
    if(outcomeApplicable){
      add('<div class="atlas17-subhead">Lectura anual de electas y no electas por sexo</div>');
      makeComp(base.filter(r=>Boolean(r.electa)),'Composición por sexo entre candidaturas electas','Cada columna suma 100% entre Mujer y Hombre solo dentro de las candidaturas electas de la población seleccionada.');
      makeComp(base.filter(r=>!Boolean(r.electa)),'Composición por sexo entre candidaturas no electas','Cada columna suma 100% entre Mujer y Hombre solo dentro de las candidaturas no electas de la población seleccionada.');
    }
    return result;
  };
})();
'''
    asset.write_text(raw + '\n' + extension + '\n', encoding='utf-8')
    print('✅ Parche V6 de claridad aplicado en', asset)
else:
    print('ℹ️ El parche V6 ya estaba aplicado; no se realizaron cambios.')



ℹ️ El parche V6 ya estaba aplicado; no se realizaron cambios.


### V7 · Visualizaciones de votos y escaños en el HTML

El notebook integra directamente la vista **«Votos y escaños»**, después de **«Padrón y escaños»**, en la aplicación HTML del Atlas. No necesita instalar archivos `.js` adicionales.

La participación del voto válido correspondiente a partidos con y sin diputaciones aparece en **un gráfico de tallos y puntos (lollipop), centrado en el porcentaje sin diputaciones, cuyo complemento es el voto con diputaciones**, no en barras ni en cantidades absolutas. Se mantienen las comparaciones entre votos y escaños del líder legislativo y del partido presidencial electo. Las tablas pueden exportarse en CSV y los gráficos en PNG/SVG.

También corrige la primera carga de «Sexo y representación», agrega hitos institucionales a las series legislativas de candidaturas electas y no electas, y reconoce nombres depurados de las siete bases Excel.


In [5]:
# V7 · INTEGRACIÓN DE REPRESENTACIÓN EN LA APLICACIÓN HTML
# Todo el JavaScript nuevo está incrustado en ESTE notebook; no se requieren
# ningún archivo JavaScript adicional. Se conserva la plantilla del Atlas.
from pathlib import Path
import re

ATLAS_V7_INLINE_JS = r'''
/* ATLAS_V7_REPRESENTACION_HTML
   Este módulo transforma las mismas tablas oficiales embebidas en el HTML.
   No depende del notebook, de imágenes estáticas, de internet ni de un servidor.
   Las tres series son nacionales; los controles Desde, Hasta y Elección de
   detalle son efectivos. La clasificación por partido con escaños se refiere
   a obtener al menos una diputación en el conjunto del país en ese año.
*/
(function(){
  'use strict';
  const sColor='#c56c4f',rColor='#137d79',sColor2='#b58a45';
  const alias={'LIBERACION NACIONAL':'PLN','UNION NACIONAL':'UN','UNIFICACION NACIONAL':'PUN','UNIDAD':'UNIDAD','UNIDAD SOCIAL CRISTIANA':'PUSC','ACCION CIUDADANA':'PAC','PROGRESO SOCIAL DEMOCRATICO':'PPSD','PUEBLO SOBERANO':'PPSO'};
  const short=p=>alias[p]||String(p).slice(0,14);
  let _cache=null;
  function makeRecords(){
    if(_cache)return _cache;
    const votes=D.votos.filter(r=>r.provincia==='Nacional'&&r.arena==='Legislativa'&&Number(r.ronda)===1);
    const seats=D.escanos.filter(r=>r.provincia==='Nacional');
    const gov=D.presidencia_asamblea;
    const byVote=new Map(),bySeat=new Map(),byGov=new Map();
    const key=(year,party)=>year+'|'+party;
    votes.forEach(r=>{const k=key(r.anio,r.partido);if(byVote.has(k))throw Error('Votos duplicados: '+k);byVote.set(k,r)});
    seats.forEach(r=>{const k=key(r.anio,r.partido);if(bySeat.has(k))throw Error('Escaños duplicados: '+k);bySeat.set(k,r)});
    gov.forEach(r=>byGov.set(r.anio,r));
    const years=[...new Set(votes.map(r=>r.anio))].sort((a,b)=>a-b);
    const totals=[];const winners=[];const presidential=[];
    for(const year of years){
      const v= votes.filter(r=>r.anio===year);
      const total=v.reduce((s,r)=>s+Number(r.votos||0),0);
      const withSeats=v.reduce((s,r)=>s+(Number(bySeat.get(key(year,r.partido))?.escanos||0)>0?Number(r.votos||0):0),0);
      const withoutSeats=total-withSeats;
      const seatsYear=seats.filter(r=>r.anio===year);
      const chamber=seatsYear.reduce((s,r)=>s+Number(r.escanos||0),0);
      if(!total||!chamber)throw Error('Falta padrón válido o magnitud en '+year);
      const row={anio:year,votos_validos:total,votos_con_escanos:withSeats,votos_sin_escanos:withoutSeats,pct_con_escanos:100*withSeats/total,pct_sin_escanos:100*withoutSeats/total};
      totals.push(row);
      const leader=[...v].sort((a,b)=>Number(b.votos)-Number(a.votos)||a.partido.localeCompare(b.partido,'es'))[0];
      const leaderSeats=Number(bySeat.get(key(year,leader.partido))?.escanos||0);
      winners.push({anio:year,partido:leader.partido,abreviatura:short(leader.partido),votos:Number(leader.votos),votos_pct:100*leader.votos/total,escanos:leaderSeats,total_escanos:chamber,escanos_pct:100*leaderSeats/chamber,brecha_pp:100*leaderSeats/chamber-100*leader.votos/total});
      const g=byGov.get(year);
      if(!g)throw Error('No hay Presidencia electa para '+year);
      const gv=byVote.get(key(year,g.partido));
      if(!gv)throw Error('No hay voto legislativo del partido presidencial electo '+year+'/'+g.partido);
      const gs=Number(bySeat.get(key(year,g.partido))?.escanos||0);
      if(Math.abs(gs-Number(g.escanos))>.001)throw Error('Inconsistencia de bancada para '+year);
      presidential.push({anio:year,presidencia:g.nombre,partido:g.partido,abreviatura:short(g.partido),votos:Number(gv.votos),votos_pct:100*gv.votos/total,escanos:gs,total_escanos:chamber,escanos_pct:100*gs/chamber,brecha_pp:100*gs/chamber-100*gv.votos/total,partido_lider_legislativo:leader.partido,coincide_lider_legislativo:g.partido===leader.partido});
      const original=D.metricas.find(m=>m.anio===year&&m.provincia==='Nacional'&&m.arena==='Legislativa'&&m.ronda===1);
      if(!original||Math.abs(Number(original.validos)-total)>.001||Math.abs(Number(original.voto_sin_escanos_pct)-row.pct_sin_escanos)>1e-6)throw Error('La serie de votos sin escaño no coincide con la validación del Atlas para '+year);
    }
    _cache={totals,winners,presidential};return _cache;
  }
  function shown(rows){return rows.filter(r=>r.anio>=state.from&&r.anio<=state.to)}
  function tracesForShares(rows){
    // Una sola variable independiente: la cuota destinada a partidos sin diputaciones.
    // La cuota con diputaciones es su complemento 100 - cuota sin diputaciones.
    // Gráfico de tallos y puntos (lollipop), NO barras apiladas ni dos líneas redundantes.
    const xs=[],ys=[];
    rows.forEach(r=>{xs.push(r.anio,r.anio,null);ys.push(0,r.pct_sin_escanos,null)});
    return [
      {type:'scatter',mode:'lines',showlegend:false,hoverinfo:'skip',
       x:xs,y:ys,line:{color:'rgba(197,108,79,0.43)',width:2.4}},
      {type:'scatter',mode:'markers',showlegend:false,name:'Partidos sin diputaciones',
       x:rows.map(r=>r.anio),y:rows.map(r=>r.pct_sin_escanos),
       customdata:rows.map(r=>[r.pct_con_escanos]),
       marker:{color:sColor,size:12,symbol:'circle',line:{color:'#ffffff',width:1.5}},
       hovertemplate:'Elección %{x}<br><b>Sin diputaciones: %{y:.2f} %</b><br>Con diputaciones: %{customdata[0]:.2f} %<extra></extra>'}
    ];
  }
  function compareTraces(rows){
    const years=rows.map(r=>r.anio), parties=rows.map(r=>r.partido);
    return [
      {type:'bar',name:'Votos legislativos válidos (%)',x:years,y:rows.map(r=>r.votos_pct),customdata:parties,marker:{color:rColor},hovertemplate:'%{x} · %{customdata}<br>Votos legislativos: %{y:.2f}%<extra></extra>'},
      {type:'bar',name:'Escaños en la Asamblea (%)',x:years,y:rows.map(r=>r.escanos_pct),customdata:parties,marker:{color:sColor2},hovertemplate:'%{x} · %{customdata}<br>Escaños: %{y:.2f}%<extra></extra>'}
    ]
  }
  function integrity(){
    const {totals,winners,presidential}=makeRecords();
    if(totals.length!==19||winners.length!==19||presidential.length!==19)throw Error('La serie debe cubrir las 19 elecciones.');
    return true;
  }
  views.representacion_voto=function(){
    const {totals,winners,presidential}=makeRecords();
    const total=shown(totals),leaders=shown(winners),gov=shown(presidential);
    const focus=total.find(r=>r.anio===state.focus),flead=leaders.find(r=>r.anio===state.focus),fgov=gov.find(r=>r.anio===state.focus);
    title('Votos y escaños','Evolución de la proporción del voto válido obtenido por partidos con y sin diputaciones y comparación de votos y escaños de las principales fuerzas.');
    note('<b>Ámbito nacional.</b> Estas comparaciones corresponden a la Asamblea Legislativa completa. Los filtros <b>Desde, Hasta y Elección de detalle</b> funcionan. El selector de provincia no cambia esta vista, porque el criterio es conseguir al menos una diputación en el país. «Sin escaños» significa sin representación partidaria directa, no ausencia de representación jurídica del electorado.');
    if(!total.length){add('<div class="empty">Sin observaciones para este intervalo.</div>');return}
    const f=focus||total[total.length-1],l=flead||leaders[leaders.length-1],g=fgov||gov[gov.length-1];
    kpis([
      ['Voto sin escaños · '+f.anio,num(f.pct_sin_escanos,2)+' %','Porcentaje de los votos válidos'],
      ['Voto con escaños · '+f.anio,num(f.pct_con_escanos,2)+' %','Porcentaje de los votos válidos'],
      ['Líder legislativo · '+l.anio,num(l.escanos,0)+' / '+num(l.total_escanos,0),esc(labelText(l.partido))+' · diputaciones'],
      ['Partido presidencial · '+g.anio,num(g.escanos_pct,2)+' %',esc(labelText(g.partido))+' · bancada']
    ]);
    if(!focus)note('La elección de detalle está fuera del intervalo. Las tarjetas muestran la última elección del periodo.',true);
    const maxShare=Math.max(...total.map(r=>r.pct_sin_escanos));
    const topShare=Math.max(5,Math.ceil((maxShare+2)/5)*5);
    plot('Voto válido destinado a partidos sin diputaciones',tracesForShares(total),total,
      {height:485,subtitle:'Nacional · '+state.from+'–'+state.to,
       xTitle:'Elección',yTitle:'Porcentaje del voto válido (%)',
       layout:{yaxis:{range:[0,topShare],title:{text:'Porcentaje del voto válido (%)'},ticksuffix:' %',dtick:5},
               showlegend:false,hovermode:'closest',margin:{l:88,r:50,t:30,b:85}},
       note:'Cada punto indica la proporción de votos válidos destinados a partidos que no obtuvieron ninguna diputación. La parte restante (100 % menos ese valor) corresponde a partidos con diputaciones. Pase el cursor por los puntos para consultar ambas proporciones.'});
    table(total,['anio','pct_con_escanos','pct_sin_escanos'],'Evolución del voto válido por acceso a diputaciones (%)');
    note('El partido legislativo más votado no coincide necesariamente con el ganador de la Presidencia. Los porcentajes de escaños se calculan sobre el total de diputaciones de cada elección.');
    plot('Líder de votos legislativos · votos frente a escaños',compareTraces(leaders),leaders,{height:510,yTitle:'Porcentaje (%)',layout:{barmode:'group',yaxis:{range:[0,90],title:{text:'Porcentaje (%)'}}},note:'Pase el cursor sobre un año para identificar el partido. Comparación de porcentaje de sufragios válidos y de escaños del mismo partido.'});
    table(leaders,['anio','partido','votos','votos_pct','escanos','total_escanos','escanos_pct','brecha_pp'],'Líder legislativo · votos y representación parlamentaria');
    note('Para el partido presidencial electo se utiliza la candidatura ganadora definitiva, incluida la segunda ronda cuando corresponde. Se comparan sus votos legislativos y sus escaños, no sus votos presidenciales.');
    plot('Partido presidencial electo · voto legislativo y bancada',compareTraces(gov),gov,{height:510,yTitle:'Porcentaje (%)',layout:{barmode:'group',yaxis:{range:[0,90],title:{text:'Porcentaje (%)'}}},note:'El ganador de la Presidencia y el líder del voto legislativo pueden ser partidos diferentes; la tabla permite identificarlos.'});
    table(gov,['anio','presidencia','partido','votos','votos_pct','escanos','total_escanos','escanos_pct','brecha_pp','partido_lider_legislativo','coincide_lider_legislativo'],'Presidencia electa · partido, votos legislativos y bancada');
    add('<div class="notice">Fuente: Atlas Electoral de Costa Rica 1953–2026. Las series utilizan votos válidos legislativos, escaños y partido presidencial electo, con denominadores nacionales consistentes.</div>');
  };
  integrity();
  window.ATLAS_V7_REPRESENTACION_HTML_OK=true;
})();

'''

def instalar_representacion_v7_en_html():
    """Actualización idempotente de los recursos que YA utiliza el notebook V7.

    No necesita archivos nuevos, no sustituye los módulos existentes, y al
    ejecutar todas las celdas la aplicación completa incluye los gráficos.
    """
    recursos = PROJECT_DIR / 'recursos'
    js_path = recursos / 'atlas_final.js'
    plantilla_path = recursos / 'atlas_template.html'
    if not js_path.is_file() or not plantilla_path.is_file():
        raise FileNotFoundError(
            'No se encontraron los recursos originales del Atlas Electoral en ' + str(recursos) +
            '. Ejecute el notebook desde la raíz del proyecto habitual (con sus siete bases y recursos).')
    js_origen = js_path.read_text(encoding='utf-8')
    marcador = 'ATLAS_V7_REPRESENTACION_HTML'
    # Sustituir el módulo V7 antiguo si ya se había ejecutado; nunca duplicar.
    if marcador in js_origen:
        patron = r'/\* ATLAS_V7_REPRESENTACION_HTML.*?window\.ATLAS_V7_REPRESENTACION_HTML_OK=true;\s*\}\)\(\);'
        nuevo, veces = re.subn(patron, lambda _: ATLAS_V7_INLINE_JS.strip(), js_origen, count=1, flags=re.S)
        if veces != 1:
            raise RuntimeError('Existe una versión anterior de V7 que no se pudo sustituir sin riesgo. No se modificó el JS.')
        js_origen = nuevo
    else:
        for exigido in ('views.representacion=', 'views.outcomes=', 'views.genero='):
            if exigido not in js_origen:
                raise RuntimeError('El archivo atlas_final.js no tiene la estructura habitual del proyecto: falta '+exigido)
        js_origen += '\n' + ATLAS_V7_INLINE_JS.strip() + '\n'
    js_path.write_text(js_origen, encoding='utf-8')
    plantilla = plantilla_path.read_text(encoding='utf-8')
    for anterior,nuevo in (
        ("['representacion','Votos y escaños']", "['representacion','Padrón y escaños']"),
        ("['representacion_voto','Voto y representación']", "['representacion_voto','Votos y escaños']"),
    ):
        plantilla = plantilla.replace(anterior,nuevo)
    opcion = "['representacion_voto','Votos y escaños'],"
    if opcion not in plantilla:
        anterior = "['representacion','Padrón y escaños'],"
        if anterior not in plantilla:
            raise RuntimeError('No se pudo localizar la entrada de representación del menú HTML original.')
        plantilla = plantilla.replace(anterior,anterior + '\n' + opcion,1)
    plantilla_path.write_text(plantilla,encoding='utf-8')
    # Parche focalizado V6: la primera carga debe utilizar la población visible del selector.
    genero = js_path.read_text(encoding='utf-8')
    genero = genero.replace("const pop=(typeof local!=='undefined' && local.sexpop)?local.sexpop:'';", 
                            "const pop=(typeof local!=='undefined' && local.sexpop)?local.sexpop:'Diputación propietaria';")
    genero = genero.replace("const result=previousGenero.apply(this,args);",
                            "local.sexpop=local.sexpop||'Diputación propietaria';\n    const result=previousGenero.apply(this,args);")
    genero = genero.replace("if(String(title).includes('a través del tiempo')){",
        "if(String(title).includes('a través del tiempo') || titleText==='Composición por sexo entre candidaturas electas' || titleText==='Composición por sexo entre candidaturas no electas'){")
    genero = genero.replace("const active=milestones.filter(m=>years.has(m.year));",
        "const active=['Diputación propietaria','Suplencia legislativa','Diputación electa'].includes(pop) ? milestones.filter(m=>years.has(m.year)) : [];")
    genero = genero.replace(
        "note:[nextOpts.note,'Líneas doradas: hitos de la normativa electoral sobre participación de mujeres. Su coincidencia con cambios en la serie es descriptiva; no demuestra causalidad por sí sola.'].filter(Boolean).join(' ')",
        "note:[nextOpts.note,active.length?'Líneas doradas: hitos de la normativa electoral relativos a candidaturas legislativas.':null].filter(Boolean).join(' ')")
    genero = genero.replace(
        "    }\n    return result;\n  };\n})();\n\n\n\n/* ATLAS_V7_REPRESENTACION_HTML",
        "    } else note(pop==='Diputación electa'?'La población seleccionada comprende solamente diputaciones electas. No corresponde dividirla en candidaturas electas y no electas.':'No se presentan gráficos de elección/no elección para esta población porque sus resultados no son comparables.');\n    return result;\n  };\n})();\n\n\n\n/* ATLAS_V7_REPRESENTACION_HTML")

    # Idempotencia de la normalización tras ejecutar varias veces en el mismo proyecto.
    genero = genero.replace("local.sexpop=local.sexpop||'Diputación propietaria';\n    local.sexpop=local.sexpop||'Diputación propietaria';\n", 
                           "local.sexpop=local.sexpop||'Diputación propietaria';\n")
    js_path.write_text(genero,encoding='utf-8')
    assert marcador in js_path.read_text(encoding='utf-8')
    assert opcion in plantilla_path.read_text(encoding='utf-8')
    print('✓ Módulo V7 de votos y escaños actualizado desde el notebook.')
    print('  Menú: Padrón y escaños · Votos y escaños. Sexo y representación corregido.')

instalar_representacion_v7_en_html()


✓ Módulo V7 de votos y escaños actualizado desde el notebook.
  Menú: Padrón y escaños · Votos y escaños. Sexo y representación corregido.


**Actualización V5** · Exportación de tarjetas como PNG/SVG sin `foreignObject`, hitos legislativos en el gráfico de género y dos tasas descriptivas junto a los flujos. El parche se instala una sola vez y es compatible con copias V2.0 sin necesidad de reemplazar fragmentos exactos de JavaScript. Para verlo, ejecute el notebook completo y abra el HTML regenerado.


## 2. Lectura e integración de las siete bases

Los vínculos usan año, provincia y denominación partidaria normalizada. Las trayectorias de personas se vinculan por nombre completo normalizado. Se conservan las diferencias documentales y no se confunde una etiqueta compartida con identidad jurídica de partidos o personas.

La base activa es `Base_Final_Candidaturas_Diputaciones_1953-2026_ENTREGA.xlsx`. La hoja `Datos` y sus 11 columnas se validan antes de procesarla. Los cambios de nombres no requieren editar el notebook; se reemplaza el Excel con el mismo nombre y estructura y se ejecuta de nuevo desde el inicio.


In [6]:
data = build_dataset(DATA_DIR, OUTPUT_DIR / "datos")
frames = {key: pd.DataFrame(rows) for key, rows in data.items() if isinstance(rows, list)}

display(pd.DataFrame(data["meta"]["fuentes"])[["base", "archivo", "bytes"]])
display(pd.DataFrame({"Tabla": list(frames), "Registros": [len(df) for df in frames.values()]}))


,base,archivo,bytes
0,leg,Base_Electoral_Diputaciones_1953-2026.xlsx,102548
1,pres,Base_Electoral_Presidencial_1953-2026.xlsx,71925
2,cand,Base_Final_Candidaturas_Diputaciones_1953-2026...,930410
3,formula,Base_Final_Candidaturas_Presidenciales_1953-20...,38190
4,electas,Base_Final_Diputaciones_Electas_1953-2026.xlsx,71918
5,diseno,Base_Final_Diseno_Papeletas_Diputaciones_1953-...,31861
6,cancelados,Base_partidos_cancelados.xlsx,65633


,Tabla,Registros
0,votos,3991
1,totales,336
2,escanos,2503
3,metricas,336
4,panel,2527
5,candidaturas,14870
6,electas,1059
7,formulas,530
8,eventos,15400
9,personas,13183


## 3. Cobertura y validación aritmética

Se comprueban los 19 procesos, las siete provincias, la suma de escaños y la coherencia de votos y padrón. En 1974 legislativo, el total de votos de la fuente excluye blancos y nulos: los emitidos analíticos son válidos + blancos + nulos, mientras `emitidos_fuente` conserva el original. Cuatro filas sin nombre se excluyen de conteos de candidaturas identificadas. La elección individual se obtiene del cruce con las declaratorias y conserva la bandera original como `electa_fuente`.

In [7]:
votos, totales, escanos = (frames[k] for k in ("votos", "totales", "escanos"))
candidaturas, electas, formulas = (frames[k] for k in ("candidaturas", "electas", "formulas"))
metricas, panel = frames["metricas"], frames["panel"]
eventos, personas = frames["eventos"], frames["personas"]

assert sorted(candidaturas.anio.unique()) == YEARS
assert set(candidaturas.provincia) == set(PROVINCES)
assert len(frames["papeletas"]) == len(YEARS) * len(PROVINCES)
assert not votos.duplicated(["anio", "provincia", "arena", "ronda", "partido"]).any()
assert np.allclose(totales.emitidos, totales.validos + totales.blancos + totales.nulos)
assert np.allclose(totales.padron, totales.emitidos + totales.abstenciones)
shares = votos.groupby(["anio", "provincia", "arena", "ronda"]).porcentaje.sum()
assert np.allclose(shares, 100)
seat_count = escanos[escanos.provincia.eq("Nacional")].groupby("anio").escanos.sum()
elected_count = electas.groupby("anio").size()
expected = pd.Series({y: 45 if y in (1953, 1958) else 57 for y in YEARS})
assert np.array_equal(seat_count.reindex(YEARS), expected)
assert np.array_equal(elected_count.reindex(YEARS), expected)
display(pd.DataFrame({"Escaños": seat_count, "Declaratorias": elected_count,
                      "Candidaturas identificadas": candidaturas.groupby("anio").size()}))
display(frames["integridad"])
print("Validación aritmética y de cobertura completada.")


,Escaños,Declaratorias,Candidaturas identificadas
anio,,,
1953,45.0,45,252
1958,45.0,45,437
1962,57.0,57,290
1966,57.0,57,311
1970,57.0,57,532
1974,57.0,57,752
1978,57.0,57,786
1982,57.0,57,647
1986,57.0,57,617


,base,anio,provincia,detalle,diferencia
0,leg,1974.0,Alajuela,Padrón frente a emitidos y abstenciones,5775.0
1,leg,1974.0,Alajuela,"Emitidos frente a válidos, blancos y nulos",-5775.0
2,leg,1974.0,Cartago,Padrón frente a emitidos y abstenciones,4488.0
3,leg,1974.0,Cartago,"Emitidos frente a válidos, blancos y nulos",-4488.0
4,leg,1974.0,Guanacaste,Padrón frente a emitidos y abstenciones,4447.0
5,leg,1974.0,Guanacaste,"Emitidos frente a válidos, blancos y nulos",-4447.0
6,leg,1974.0,Heredia,Padrón frente a emitidos y abstenciones,2262.0
7,leg,1974.0,Heredia,"Emitidos frente a válidos, blancos y nulos",-2262.0
8,leg,1974.0,Limón,Padrón frente a emitidos y abstenciones,2060.0
9,leg,1974.0,Limón,"Emitidos frente a válidos, blancos y nulos",-2060.0


Validación aritmética y de cobertura completada.


## 4. Participación y abstencionismo

Las tasas nacionales se calculan sumando numeradores y denominadores provinciales. La comparación entre arenas usa la primera ronda presidencial. La segunda ronda se conserva por separado. La cobertura son las siete provincias; no se agrega voto exterior.

In [8]:
nacional = metricas[metricas.provincia.eq("Nacional") & metricas.ronda.eq(1)].sort_values("anio")
legislativa = nacional[nacional.arena.eq("Legislativa")].copy()
provincial = metricas[metricas.provincia.ne("Nacional") & metricas.ronda.eq(1) & metricas.arena.eq("Legislativa")]
show(px.line(nacional, x="anio", y="abstencion_pct", color="arena", markers=True,
             labels={"anio": "Elección", "abstencion_pct": "Abstencionismo (%)", "arena": "Elección"}),
     "Participación histórica: abstencionismo presidencial y legislativo")
heat = provincial.pivot(index="provincia", columns="anio", values="abstencion_pct").reindex(PROVINCES)
show(go.Figure(go.Heatmap(x=heat.columns, y=heat.index, z=heat.values,
    colorscale=[[0,"#edf6f4"],[1,"#007f7a"]], colorbar=dict(title="%"))),
    "Abstencionismo legislativo por provincia y año")
rounds = metricas[metricas.provincia.eq("Nacional") & metricas.arena.eq("Presidencial")].copy()
rounds["Ronda"] = rounds.ronda.map({1: "Primera ronda", 2: "Segunda ronda"})
display(rounds[["anio", "Ronda", "padron", "emitidos", "abstencion_pct"]])


,anio,Ronda,padron,emitidos,abstencion_pct
11,1953,Primera ronda,293670.0,197489.0,32.751388
27,1958,Primera ronda,354780.0,229543.0,35.299904
43,1962,Primera ronda,483980.0,391406.0,19.127650
59,1966,Primera ronda,554627.0,451490.0,18.595741
75,1970,Primera ronda,675285.0,562766.0,16.662446
91,1974,Primera ronda,875043.0,699340.0,20.079356
107,1978,Primera ronda,1058445.0,860206.0,18.729268
123,1982,Primera ronda,1261127.0,991679.0,21.365652
139,1986,Primera ronda,1486474.0,1216300.0,18.175494
155,1990,Primera ronda,1692050.0,1384326.0,18.186460


## 5. Voto válido, blanco y nulo

Blancos y nulos se dividen entre votos emitidos; el abstencionismo se divide entre padrón. Son fenómenos diferentes. Los filtros del HTML permiten cambiar el indicador y comparar provincias.

In [9]:
no_validos = nacional.melt(id_vars=["anio", "arena"], value_vars=["blancos_pct", "nulos_pct"],
    var_name="Tipo", value_name="Porcentaje")
show(px.line(no_validos, x="anio", y="Porcentaje", color="arena", line_dash="Tipo", markers=True),
     "Votos blancos y nulos como proporción de los emitidos")
display(nacional[["anio", "arena", "blancos", "nulos", "invalidos_pct"]].tail(12))


,anio,arena,blancos,nulos,invalidos_pct
226,2006,Legislativa,14652.0,34286.0,2.942933
227,2006,Presidencial,8834.0,30422.0,2.360201
243,2010,Presidencial,6959.0,32555.0,2.025479
242,2010,Legislativa,20077.0,30806.0,2.608438
263,2014,Legislativa,13053.0,34801.0,2.282942
264,2014,Presidencial,5724.0,37969.0,2.084144
287,2018,Legislativa,12653.0,39746.0,2.390854
288,2018,Presidencial,5090.0,22937.0,1.286626
311,2022,Legislativa,18431.0,23199.0,1.965996
312,2022,Presidencial,11599.0,17636.0,1.380449


## 6. Concentración, fragmentación y competencia

NEP = 1 / suma de proporciones de voto al cuadrado. Las concentraciones del primero y de los dos primeros permiten examinar predominio y bipartidismo sin imponer periodizaciones. La volatilidad compara denominaciones nominales, no continuidad jurídica.

In [10]:
show(px.line(nacional, x="anio", y="nep", color="arena", markers=True,
             labels={"nep": "Número efectivo de partidos", "anio": "Elección"}),
     "Fragmentación en las arenas presidencial y legislativa")
concentration = legislativa.melt(id_vars=["anio"], value_vars=["concentracion_1", "concentracion_2"],
                                var_name="Indicador", value_name="Porcentaje")
show(px.line(concentration, x="anio", y="Porcentaje", color="Indicador", markers=True),
     "Concentración legislativa del primero y los dos primeros")
display(nacional[["anio", "arena", "ganador", "partidos_con_votos", "nep", "margen", "volatilidad_nominal"]].tail(12))


,anio,arena,ganador,partidos_con_votos,nep,margen,volatilidad_nominal
226,2006,Legislativa,LIBERACION NACIONAL,27,4.627774,11.196119,27.582924
227,2006,Presidencial,LIBERACION NACIONAL,14,2.981362,1.118786,36.002345
243,2010,Presidencial,LIBERACION NACIONAL,9,3.044466,21.850667,21.157840
242,2010,Legislativa,LIBERACION NACIONAL,18,4.780011,19.654810,19.624107
263,2014,Legislativa,LIBERACION NACIONAL,20,6.225324,2.224380,26.789220
264,2014,Presidencial,ACCION CIUDADANA,13,4.369871,0.916914,28.319710
287,2018,Legislativa,LIBERACION NACIONAL,27,7.672039,1.332900,35.959473
288,2018,Presidencial,RESTAURACION NACIONAL,13,5.512805,3.418955,49.040702
311,2022,Legislativa,LIBERACION NACIONAL,36,7.987608,9.784370,51.699259
312,2022,Presidencial,LIBERACION NACIONAL,25,6.114215,10.503152,64.914056


## 7. Relación presidencial–legislativa

La brecha es porcentaje presidencial menos legislativo, en puntos porcentuales. La divergencia compara las distribuciones completas. Estos agregados no identifican cuántas personas cruzaron su voto.

In [11]:
panel_nacional = panel[panel.provincia.eq("Nacional")]
ultimo = panel_nacional[panel_nacional.anio.eq(2026) & panel_nacional.brecha_pp.notna()].sort_values("brecha_pp")
show(px.bar(ultimo, x="brecha_pp", y="partido", orientation="h",
             labels={"brecha_pp": "Presidencial − legislativo (pp)", "partido": "Partido"}),
     "Brecha entre arenas por partido · 2026")
show(px.line(legislativa, x="anio", y="divergencia_arenas", markers=True,
             labels={"divergencia_arenas": "Divergencia (pp)"}),
     "Divergencia histórica entre distribuciones presidenciales y legislativas")


## 8. Votos, escaños y desproporcionalidad

Gallagher = raíz de la mitad de la suma de diferencias al cuadrado entre porcentajes de voto y escaños. Se utilizan distribuciones completas. La magnitud provincial permite analizar las circunscripciones sin asumir que todos los partidos presentaron listas completas.

In [12]:
representation = legislativa[["anio", "nep", "nep_escanos"]].melt(id_vars="anio", var_name="Distribución", value_name="NEP")
show(px.line(representation, x="anio", y="NEP", color="Distribución", markers=True),
     "Fragmentación de votos y escaños")
show(px.line(legislativa, x="anio", y="gallagher", markers=True,
             labels={"gallagher": "Gallagher"}), "Desproporcionalidad legislativa")
display(panel_nacional[panel_nacional.anio.eq(2026)][["partido", "porcentaje_Legislativa", "escanos", "escanos_pct", "desviacion_pp"]].sort_values("escanos", ascending=False))


,partido,porcentaje_Legislativa,escanos,escanos_pct,desviacion_pp
2474,PUEBLO SOBERANO,45.199059,31.0,54.385965,9.186906
2469,LIBERACION NACIONAL,23.249153,17.0,29.824561,6.575408
2466,FRENTE AMPLIO,12.334543,7.0,12.280702,-0.053842
2475,UNIDAD SOCIAL CRISTIANA,4.524633,1.0,1.754386,-2.770248
2456,AGENDA CIUDADANA,4.069753,1.0,1.754386,-2.315367
2455,ACTUEMOS YA,0.495633,0.0,0.000000,-0.495633
2468,JUSTICIA SOCIAL COSTARRICENSE,0.311048,0.0,0.000000,-0.311048
2477,UNION COSTARRICENSE DEMOCRATICA,0.183681,0.0,0.000000,-0.183681
2476,UNIDOS PODEMOS,1.195500,0.0,0.000000,-1.195500
2473,PROGRESO SOCIAL DEMOCRATICO,0.582875,0.0,0.000000,-0.582875


## 8.1. Voto válido y representación partidaria

Se calculan las proporciones de votos válidos correspondientes a organizaciones que obtuvieron al menos una diputación y a aquellas que no obtuvieron ninguna. Se trata de **representación partidaria directa de la opción votada**, no de representación jurídica individual.

Las tres series nuevas son: (1) porcentaje del voto válido a partidos con y sin diputaciones, mediante puntos y tallos proporcionales, sin barras apiladas ni líneas redundantes; (2) porcentaje de votos legislativos frente a porcentaje de escaños del partido con más votos legislativos; y (3) las mismas dos proporciones para el partido que ganó definitivamente la Presidencia. **No se presenta una gráfica de cantidades absolutas de votos.**

Para el cálculo se utilizan registros nacionales, sin sumar otra vez las provincias. Se cotejan los votos válidos y las bancadas con las tablas originales. El notebook genera PNG de alta resolución y CSV verificables en `salidas/graficas_articulo/`; el HTML recrea las series con filtros y descargas.


In [13]:
"""V7: controles de votos válidos, escaños y lecturas interactivas en el notebook.
El Atlas público consume los datos de salidas/datos; esta celda NO crea PNG,
figuras para el artículo, CSV independientes ni carpetas adicionales.
"""
import pandas as pd
import numpy as np
import plotly.graph_objects as go

REPRESENTADO = "#206E88"
SIN_ESCANO = "#D1744A"
ESCANOS = "#A98535"

def calcular_representacion_v7(frames):
    """Usa SOLO filas Nacional para no duplicar agregados provinciales.

    Define 'con escaños' cuando el partido obtuvo >= 1 diputación en cualquier
    provincia en esa misma elección. Eso NO identifica votantes individuales,
    preferencias personales ni escaños específicos para cada sufragio.
    """
    v = frames['votos'].copy()
    s = frames['escanos'].copy()
    v = v.loc[v.provincia.eq('Nacional') & v.arena.eq('Legislativa') & v.ronda.eq(1),
              ['anio', 'partido', 'votos']].copy()
    s = s.loc[s.provincia.eq('Nacional'), ['anio', 'partido', 'escanos']].copy()
    v['anio'] = v.anio.astype(int)
    s['anio'] = s.anio.astype(int)
    assert not v.duplicated(['anio', 'partido']).any(), 'Votos legislativos duplicados por partido/año'
    assert not s.duplicated(['anio', 'partido']).any(), 'Escaños duplicados por partido/año'
    v['votos'] = pd.to_numeric(v.votos, errors='raise').fillna(0)
    s['escanos'] = pd.to_numeric(s.escanos, errors='raise').fillna(0)
    assert (v.votos >= 0).all() and (s.escanos >= 0).all()
    v = v.merge(s, on=['anio', 'partido'], how='left', validate='one_to_one', indicator=True)
    # Sin fila de escaño == sin representación; nunca excluir del denominador.
    v['escanos'] = v.escanos.fillna(0)
    v['con_escanos'] = v.escanos.gt(0)
    assert v.anio.nunique() == 19 and v.anio.min() == 1953 and v.anio.max() == 2026

    base = v.groupby('anio', as_index=False).agg(votos_validos=('votos','sum'))
    con = v[v.con_escanos].groupby('anio').votos.sum().rename('votos_con_escanos')
    sin = v[~v.con_escanos].groupby('anio').votos.sum().rename('votos_sin_escanos')
    base = base.join(con, on='anio').join(sin, on='anio').fillna({'votos_con_escanos':0, 'votos_sin_escanos':0})
    base['pct_con_escanos'] = 100*base.votos_con_escanos/base.votos_validos
    base['pct_sin_escanos'] = 100*base.votos_sin_escanos/base.votos_validos
    base['partidos_con_escanos'] = v[v.con_escanos].groupby('anio').partido.nunique().reindex(base.anio,fill_value=0).to_numpy()
    base['partidos_sin_escanos'] = v[(~v.con_escanos)&v.votos.gt(0)].groupby('anio').partido.nunique().reindex(base.anio,fill_value=0).to_numpy()
    assert np.allclose(base.votos_con_escanos+base.votos_sin_escanos,base.votos_validos)
    assert np.allclose(base.pct_con_escanos+base.pct_sin_escanos,100)
    if 'totales' in frames:
        validos = frames['totales'].query('provincia == "Nacional" and arena == "Legislativa" and ronda == 1')
        chk = base.merge(validos[['anio','validos']],on='anio',validate='one_to_one')
        assert np.allclose(chk.votos_validos,chk.validos), 'Los votos por partido no suman los votos válidos oficiales'
    if 'metricas' in frames:
        metr = frames['metricas'].query('provincia == "Nacional" and arena == "Legislativa" and ronda == 1')
        chk = base.merge(metr[['anio','voto_sin_escanos_pct']],on='anio',validate='one_to_one')
        assert np.allclose(chk.pct_sin_escanos,chk.voto_sin_escanos_pct,atol=1e-8), 'Difiere del indicador original'

    escanos_total = s.groupby('anio').escanos.sum().rename('total_escanos').reset_index()
    # Partido legislativo ganador por pluralidad de votos legislativos, NO por Presidencia.
    lider = v.sort_values(['anio','votos','partido'],ascending=[True,False,True]).drop_duplicates('anio').copy()
    lider = lider.merge(base[['anio','votos_validos']],on='anio',validate='one_to_one').merge(escanos_total,on='anio',validate='one_to_one')
    lider['votos_pct'] = 100*lider.votos/lider.votos_validos
    lider['escanos_pct'] = 100*lider.escanos/lider.total_escanos
    lider['brecha_escanos_votos_pp'] = lider.escanos_pct-lider.votos_pct
    lider = lider[['anio','partido','votos','votos_pct','escanos','total_escanos','escanos_pct','brecha_escanos_votos_pp']].sort_values('anio')

    # Ganador definitivo presidencial: incluye resolución de segunda vuelta.
    gov = frames['presidencia_asamblea'][['anio','nombre','partido','escanos','escanos_pct']].copy()
    gov['anio'] = gov.anio.astype(int)
    assert not gov.duplicated('anio').any()
    pres = gov.merge(v[['anio','partido','votos']],on=['anio','partido'],how='left',validate='one_to_one')
    assert pres.votos.notna().all(), 'Ganador presidencial sin votos legislativos correspondientes'
    pres = pres.merge(base[['anio','votos_validos']],on='anio',validate='one_to_one').merge(escanos_total,on='anio',validate='one_to_one')
    pres['votos_pct'] = 100*pres.votos/pres.votos_validos
    pres['escanos_pct_verificado'] = 100*pres.escanos/pres.total_escanos
    assert np.allclose(pres.escanos_pct,pres.escanos_pct_verificado,atol=1e-8)
    pres['brecha_escanos_votos_pp'] = pres.escanos_pct-pres.votos_pct
    pres = pres.merge(lider[['anio','partido']].rename(columns={'partido':'ganador_legislativo'}),on='anio',validate='one_to_one')
    pres['coincide_ganador_legislativo'] = pres.partido.eq(pres.ganador_legislativo)

    # Primer lugar en PRIMERA vuelta, distinto del presidente definitivo en algunos años.
    prep = frames['votos'].copy()
    prep = prep.loc[prep.provincia.eq('Nacional') & prep.arena.eq('Presidencial') & prep.ronda.eq(1),['anio','partido','votos']]
    first = prep.sort_values(['anio','votos','partido'],ascending=[True,False,True]).drop_duplicates('anio')[['anio','partido']]
    first = first.rename(columns={'partido':'primer_lugar_primera_ronda_presidencial'})
    pres = pres.merge(first,on='anio',validate='one_to_one')
    pres['coincide_primera_ronda'] = pres.partido.eq(pres.primer_lugar_primera_ronda_presidencial)
    pres = pres[['anio','nombre','partido','votos','votos_pct','escanos','total_escanos','escanos_pct','brecha_escanos_votos_pp',
                 'ganador_legislativo','coincide_ganador_legislativo','primer_lugar_primera_ronda_presidencial','coincide_primera_ronda']].sort_values('anio')
    if 'concordancia_gobierno' in frames:
        chk = pres.merge(frames['concordancia_gobierno'][['anio','ganador_presidencial','ganador_voto_legislativo']],on='anio')
        assert (chk.partido == chk.ganador_presidencial).all() and (chk.ganador_legislativo == chk.ganador_voto_legislativo).all()
    assert len(base)==len(lider)==len(pres)==19
    return base, lider, pres

def figuras_interactivas_v7(base,lider,pres):
    """Tres figuras Plotly compatibles con el método show() del notebook."""
    years=base.anio.astype(int).tolist()
    vals=base.pct_sin_escanos.to_numpy(dtype=float)
    stems_x=[];stems_y=[]
    for yr,valor in zip(years,vals):
        stems_x.extend([yr,yr,None]);stems_y.extend([0,float(valor),None])
    f1=go.Figure()
    f1.add_trace(go.Scatter(x=stems_x,y=stems_y,mode='lines',showlegend=False,
                            hoverinfo='skip',line=dict(color='rgba(209,116,74,.45)',width=2.5)))
    f1.add_trace(go.Scatter(x=years,y=vals,mode='markers',showlegend=False,
                            marker=dict(color=SIN_ESCANO,size=12,line=dict(color='white',width=1.5)),
                            customdata=np.column_stack((base.pct_con_escanos.to_numpy(dtype=float),)),
                            hovertemplate='Elección %{x}<br><b>Sin diputaciones: %{y:.2f} %</b><br>Con diputaciones: %{customdata[0]:.2f} %<extra></extra>'))
    upper=max(5,float(np.ceil((vals.max()+2)/5)*5))
    f1.update_layout(height=485,hovermode='closest',showlegend=False,
                     title='Voto válido destinado a partidos sin diputaciones',
                     xaxis=dict(title='Elección',tickmode='array',tickvals=years,tickangle=-45),
                     yaxis=dict(title='Porcentaje del voto válido (%)',range=[0,upper],ticksuffix=' %',dtick=5),
                     margin=dict(l=82,r=45,t=65,b=92))
    def compare(df,label):
        fig=go.Figure()
        extra=np.array(df.partido).reshape(-1,1)
        for key,name,col in [('votos_pct','Votos legislativos válidos (%)',REPRESENTADO),('escanos_pct','Escaños (%)',ESCANOS)]:
            fig.add_trace(go.Bar(x=df.anio.astype(str),y=df[key],name=name,marker_color=col,customdata=extra,
                        hovertemplate='%{x} · %{customdata[0]}<br>%{y:.2f}%<extra>'+name+'</extra>'))
        fig.update_layout(barmode='group',height=520,title=label,hovermode='x unified',yaxis_title='Porcentaje (%)')
        return fig
    f2=compare(lider,'Partido más votado en diputaciones: voto válido y cuota de escaños')
    f3=compare(pres,'Partido presidencial electo: voto válido legislativo y cuota de escaños')
    return [f1,f2,f3]


def producir_v7(frames, show_fig=None, display_tables=False):
    base, lider, pres = calcular_representacion_v7(frames)
    if show_fig is not None:
        for figura in figuras_interactivas_v7(base, lider, pres):
            show_fig(figura)
    if display_tables:
        print('Voto válido por acceso a diputaciones, 2022 y 2026:')
        display(base.loc[base.anio.isin([2022,2026]), ['anio','pct_con_escanos','pct_sin_escanos']])
        print('Años en que el partido presidencial electo no encabezó el voto legislativo:')
        display(pres.loc[~pres.coincide_ganador_legislativo,
                ['anio','partido','ganador_legislativo','votos_pct','escanos_pct']])
    return {'por_anio':base,'lider_legislativo':lider,'partido_presidencial':pres}

V7_REPRESENTACION = producir_v7(frames, show_fig=show, display_tables=True)
assert len(V7_REPRESENTACION['por_anio']) == 19
print('Comprobaciones V7 concluidas. Sin exportaciones editoriales adicionales.')


Voto válido por acceso a diputaciones, 2022 y 2026:


,anio,pct_con_escanos,pct_sin_escanos
17,2022,78.733226,21.266774
18,2026,89.377142,10.622858


Años en que el partido presidencial electo no encabezó el voto legislativo:


,anio,partido,ganador_legislativo,votos_pct,escanos_pct
1,1958,UNION NACIONAL,LIBERACION NACIONAL,21.366383,22.222222
3,1966,UNIFICACION NACIONAL,LIBERACION NACIONAL,43.159060,45.614035
15,2014,ACCION CIUDADANA,LIBERACION NACIONAL,23.481363,22.807018
16,2018,ACCION CIUDADANA,LIBERACION NACIONAL,16.249284,17.543860
17,2022,PROGRESO SOCIAL DEMOCRATICO,LIBERACION NACIONAL,15.035609,17.543860


Comprobaciones V7 concluidas. Sin exportaciones editoriales adicionales.


### Registro de decisiones para la próxima revisión del artículo (sin modificar el Word todavía)

- Conservar la gráfica histórica de **número de partidos con votos y partidos con escaños**. Añadir las tres visualizaciones de V7 en su bloque de oferta, votos y representación; la numeración deberá ajustarse **después** de decidir cuáles serán las figuras definitivas.
- En el orden de la sección de participación, intercambiar la posición actual de las figuras **5 y 6**: el **ranking provincial** debe seguir inmediatamente al gráfico de **abstencionismo provincial**, y luego debe aparecer la **distribución del padrón**. No hacerlo todavía en el Word.
- Explicar con precisión la dominancia provincial del PLN entre 1953 y 1974: encabeza **41 de las 42** combinaciones provincia–elección, con excepción de **Puntarenas, 1966** (Unificación Nacional). Es predominio en pluralidades legislativas provinciales, no ausencia de otros partidos ni partido único.
- En representación femenina, separar **candidaturas propietarias** (salto a partir de 2002), **suplencias** (variaciones e incremento desde fines de los noventa; 1962 no está documentado por categoría), **diputaciones efectivamente electas** (desde 2018 superan el 40 %) y **fórmulas presidenciales** (interpretación condicionada por denominadores reducidos y diferencias entre Presidencia y vicepresidencias). Los hitos normativos se identifican como coincidencias temporales, no como efectos causales estimados.
- En la redacción del artículo, citar cada nueva figura junto al párrafo que la interpreta y explicar expresamente la fórmula y los denominadores. Evitar el uso ambiguo de «votos sin representación»; preferir «votos a partidos sin escaños».


## 9. Presidencia y Asamblea

El partido ganador se identifica por la fórmula presidencial electa, no por el primer lugar en primera ronda. La mayoría se refiere a la composición inicial de la Asamblea. Los antecedentes y la elección siguiente se extraen del corpus; no describen cambios de bancada durante el mandato.

In [14]:
gobierno = frames["presidencia_asamblea"]
show(px.bar(gobierno, x="anio", y="escanos_pct", color="mayoria_inicial",
             hover_data=["nombre", "partido", "escanos"],
             labels={"escanos_pct": "Escaños del partido presidencial (%)", "mayoria_inicial": "Mayoría inicial"}),
     "Bancada inicial del partido presidencial electo")
display(gobierno[["anio", "nombre", "partido", "escanos", "mayoria_inicial", "antecedente_legislativo_electo", "eleccion_siguiente", "cambio_voto_legislativo_pp", "cambio_escanos"]])


,anio,nombre,partido,escanos,mayoria_inicial,antecedente_legislativo_electo,eleccion_siguiente,cambio_voto_legislativo_pp,cambio_escanos
0,1953,José Figueres Ferrer,LIBERACION NACIONAL,30.0,True,False,1958.0,-23.066848,-10.0
1,1958,Mario Echandi Jiménez,UNION NACIONAL,10.0,False,True,1962.0,-8.095996,-1.0
2,1962,Francisco José Orlich Bolmarcich,LIBERACION NACIONAL,29.0,True,True,1966.0,0.081978,0.0
3,1966,José Joaquín Trejos Fernández,UNIFICACION NACIONAL,26.0,False,False,1970.0,-7.265767,-4.0
4,1970,José Figueres Ferrer,LIBERACION NACIONAL,32.0,True,False,1974.0,-9.836745,-5.0
5,1974,Daniel Oduber Quirós,LIBERACION NACIONAL,27.0,False,True,1978.0,-2.020279,-2.0
6,1978,Rodrigo Carazo Odio,UNIDAD,27.0,False,True,1982.0,-14.331614,-9.0
7,1982,Luis Alberto Monge Álvarez,LIBERACION NACIONAL,33.0,True,True,1986.0,-7.317600,-4.0
8,1986,Óscar Arias Sánchez,LIBERACION NACIONAL,29.0,True,True,1990.0,-5.949433,-4.0
9,1990,Rafael Ángel Calderón Fournier,UNIDAD SOCIAL CRISTIANA,29.0,True,True,1994.0,-5.835341,-4.0


## 10. Permanencia partidaria y alcance territorial

Aparición y desaparición se refieren al corpus electoral, mientras inscripción y cancelación pertenecen al registro jurídico. La cobertura efectiva de candidaturas y votos no reemplaza la escala legal de inscripción. Un nombre compartido no garantiza una organización idéntica.

In [15]:
historia_partidos = frames["historia_partidos"]
alcance = frames["alcance_partidos"]
display(historia_partidos.sort_values("elecciones_observadas", ascending=False).head(20))
coverage = alcance[alcance.anio.eq(2026)].sort_values("provincias_con_candidaturas")
show(px.bar(coverage, x="provincias_con_candidaturas", y="partido", orientation="h",
             hover_data=["oferta_presidencial", "candidaturas", "dispersion_apoyo_pp"]),
     "Cobertura provincial de candidaturas legislativas · 2026")


,partido,primera_aparicion,ultima_aparicion,elecciones_observadas,primera_diputacion,escanos_acumulados,denominacion
63,LIBERACION NACIONAL,1953,2026,19,1953.0,467.0,Etiqueta nominal; continuidad jurídica no esta...
115,UNIDAD SOCIAL CRISTIANA,1986,2026,11,1986.0,163.0,Etiqueta nominal; continuidad jurídica no esta...
119,UNION AGRICOLA CARTAGINES,1970,2010,11,1974.0,5.0,Etiqueta nominal; continuidad jurídica no esta...
55,INDEPENDIENTE,1958,1998,8,1958.0,3.0,Etiqueta nominal; continuidad jurídica no esta...
57,INTEGRACION NACIONAL,1998,2026,8,1998.0,5.0,Etiqueta nominal; continuidad jurídica no esta...
23,AUTENTICO LIMONENSE,1978,2018,7,NaN,0.0,Etiqueta nominal; continuidad jurídica no esta...
69,MOVIMIENTO LIBERTARIO,1998,2022,7,1998.0,26.0,Etiqueta nominal; continuidad jurídica no esta...
1,ACCION CIUDADANA,2002,2022,6,2002.0,65.0,Etiqueta nominal; continuidad jurídica no esta...
46,FRENTE AMPLIO,2006,2026,6,2006.0,25.0,Etiqueta nominal; continuidad jurídica no esta...
95,RENOVACION COSTARRICENSE,1998,2018,6,1998.0,5.0,Etiqueta nominal; continuidad jurídica no esta...


## 11. Tamaño de las listas, propietarios y suplencias

Los 45 escaños de 1953 y 1958 y los 57 desde 1962 son magnitudes nacionales. No son un número obligatorio de candidaturas por partido: una organización puede presentar listas parciales o participar en ciertas provincias. Las filas sin nombre no cuentan como candidaturas identificadas.

In [16]:
oferta = candidaturas.groupby(["anio", "tipo"]).size().reset_index(name="Candidaturas")
show(px.line(oferta, x="anio", y="Candidaturas", color="tipo", markers=True),
     "Evolución de las candidaturas propietarias y suplencias")
display(panel[panel.anio.eq(2026) & panel.provincia.ne("Nacional")][["provincia", "partido", "propietarias", "suplencias", "magnitud", "cobertura_lista_pct"]].head(21))


,provincia,partido,propietarias,suplencias,magnitud,cobertura_lista_pct
2335,Alajuela,ACTUEMOS YA,NaN,NaN,12.0,NaN
2336,Alajuela,AGENDA CIUDADANA,12.0,3.0,12.0,100.000000
2337,Alajuela,ALIANZA COSTA RICA PRIMERO,12.0,3.0,12.0,100.000000
2338,Alajuela,ANTICORRUPCION COSTARRICENSE,NaN,NaN,12.0,NaN
2339,Alajuela,AQUI COSTA RICA MANDA,6.0,1.0,12.0,50.000000
2340,Alajuela,AVANZA,3.0,0.0,12.0,25.000000
2341,Alajuela,CENTRO DEMOCRATICO Y SOCIAL,10.0,3.0,12.0,83.333333
2342,Alajuela,COMPATRIOTAS,NaN,NaN,12.0,NaN
2343,Alajuela,DE LA CLASE TRABAJADORA,3.0,0.0,12.0,25.000000
2344,Alajuela,ESPERANZA NACIONAL,12.0,0.0,12.0,100.000000


## 12. Sexo, listas y acceso a la representación

Se compara la participación de mujeres entre propietarias, suplencias, encabezamientos y personas electas. Sexo procede de las bases aportadas. La asociación con cambios normativos necesita una cronología jurídica documentada; este notebook no atribuye efectos causales a reformas no incorporadas.

In [17]:
genero = frames["genero_renovacion"]
genero_nacional = genero[genero.provincia.eq("Nacional")]
gender_long = genero_nacional.melt(id_vars="anio", value_vars=["mujeres_propietarias_pct", "mujeres_suplentes_pct", "mujeres_encabezamientos_pct", "mujeres_electas_pct"], var_name="Categoría", value_name="Mujeres (%)")
show(px.line(gender_long, x="anio", y="Mujeres (%)", color="Categoría", markers=True),
     "De las listas a la representación: mujeres a lo largo del tiempo")
owners = candidaturas[candidaturas.tipo.eq("Propietaria")]
success = owners.groupby(["anio", "sexo"]).agg(candidaturas=("nombre", "size"), electas=("electa", "sum")).reset_index()
success["eleccion_pct"] = 100 * success.electas / success.candidaturas
show(px.line(success, x="anio", y="eleccion_pct", color="sexo", markers=True,
             labels={"eleccion_pct": "Candidaturas electas (%)", "sexo": "Sexo según fuente"}),
     "Acceso al escaño entre candidaturas propietarias por sexo")


## 13. Recurrencias, experiencia y movilidad

La experiencia es un antecedente electoral registrado, no experiencia política total. En 1953 los historiales están truncados por el inicio del corpus. Se distingue cambio de partido, provincia o cargo entre elecciones de cambios de bancada dentro del mandato. Coincidencias nominales no verifican exhaustivamente homónimos.

In [18]:
display(personas.sort_values(["elecciones", "candidaturas"], ascending=False).head(20))
show(px.line(genero_nacional, x="anio", y="experiencia_pct", markers=True,
             labels={"experiencia_pct": "Personas con antecedente electoral (%)"}),
     "Experiencia previa entre personas candidatas a diputaciones")
movimientos = frames["movimientos"]
display(movimientos.groupby(["dimension", "origen", "destino"]).size().reset_index(name="Cambios").sort_values("Cambios", ascending=False).head(25))
carreras = eventos[eventos.cargo.ne("Diputación") & eventos.experiencia_previa]
display(carreras[["anio", "nombre", "partido", "cargo", "participaciones_previas", "electa"]].head(20))


,id_persona,nombre,sexo,inicio,fin,elecciones,candidaturas,partidos,provincias,cargos,elecciones_exitosas,primera_eleccion,intentos_hasta_eleccion,identidad,nombre_original
6769,8551afd385b0c0,Walter Muñoz Céspedes,Hombre,1998,2022,7,10,1,1,2,2,1998.0,1.0,Coincidencia nominal exacta,Walter Muñoz Céspedes
1262,193c4d408f9bc2,Otto Guevara Guth,Hombre,1998,2022,7,8,2,1,2,2,1998.0,1.0,Coincidencia nominal exacta,Otto Guevara Guth
1133,1694875a5cc529,Juan Guillermo Brenes Castillo,Hombre,1966,2002,6,6,2,1,1,3,1974.0,3.0,Coincidencia nominal exacta,Juan Guillermo Brenes Castillo
9494,ba0847319cbed4,Marvin Luis Wright Lindo,Hombre,1978,1998,6,6,1,1,1,0,NaN,NaN,Coincidencia nominal exacta,Marvin Luis Wright Lindo
9572,bba4e9b6a76a46,Edgar Mora Garcia,Hombre,1953,1982,6,6,5,1,1,1,1953.0,1.0,Coincidencia nominal exacta,Edgar Mora Garcia
7587,94ae1e2160e8bd,Óscar Andrés López Arias,Hombre,2006,2022,5,7,1,1,2,2,2006.0,1.0,Coincidencia nominal exacta,Óscar Andrés López Arias
7764,98427373dde399,Álvaro González Espinoza,Hombre,1974,1998,5,6,3,2,2,0,NaN,NaN,Coincidencia nominal exacta,Álvaro González Espinoza
7982,9cdb1e0ff6badd,Rolando Araya Monge,Hombre,1974,2022,5,6,4,1,2,1,1974.0,1.0,Coincidencia nominal exacta,Rolando Araya Monge
145,035fa7971bcb1d,Ottón Solís Fallas,Hombre,1994,2014,5,5,2,1,2,2,1994.0,1.0,Coincidencia nominal exacta,Ottón Solís Fallas
744,0ecc65a6358862,José Angel Jara Chavarria,Hombre,1953,1994,5,5,4,1,1,1,1958.0,2.0,Coincidencia nominal exacta,José Angel Jara Chavarria


,dimension,origen,destino,Cambios
0,Cargo,Diputación,Presidencia,45
170,Partido,INDEPENDIENTE,INDEPENDIENTE OBRERO,29
386,Partido,UNIDAD,UNIDAD SOCIAL CRISTIANA,27
1,Cargo,Diputación,Primera Vicepresidencia,26
511,Provincia,San José,Alajuela,25
490,Provincia,Cartago,San José,25
484,Provincia,Alajuela,San José,23
499,Provincia,Heredia,San José,21
2,Cargo,Diputación,Segunda Vicepresidencia,21
512,Provincia,San José,Cartago,18


,anio,nombre,partido,cargo,participaciones_previas,electa
420,1958,Francisco José Orlich Bolmarcich,LIBERACION NACIONAL,Presidencia,1,False
517,1958,José Joaquín Peralta Esquivel,UNION NACIONAL,Primera Vicepresidencia,1,True
579,1958,Mario Echandi Jiménez,UNION NACIONAL,Presidencia,1,True
624,1958,Otto Cortés Fernández,LIBERACION NACIONAL,Primera Vicepresidencia,1,False
646,1958,Rafael París Steffens,LIBERACION NACIONAL,Segunda Vicepresidencia,1,False
796,1962,Enrique Obregón Valverde,ACCION DEMOCRATICA POPULAR,Presidencia,1,False
811,1962,Francisco José Orlich Bolmarcich,LIBERACION NACIONAL,Presidencia,2,True
914,1962,Manuel Escalante Durán,REPUBLICANO NACIONAL,Primera Vicepresidencia,1,False
949,1962,Otilio Ulate Blanco,UNION NACIONAL,Presidencia,1,False
967,1962,Rafael Ángel Calderón Guardia,REPUBLICANO NACIONAL,Presidencia,1,False


## 14. Fórmulas presidenciales, vicepresidencias y apellidos

El HTML permite consultar fórmulas y trayectorias individuales. Los apellidos se toman de campos expresamente separados de las bases de electas y fórmulas; su repetición no demuestra parentesco ni familias políticas.

In [19]:
composition = formulas.groupby(["anio", "cargo"]).sexo.agg(lambda s: 100 * s.eq("Mujer").mean()).reset_index(name="Mujeres (%)")
show(px.line(composition, x="anio", y="Mujeres (%)", color="cargo", markers=True),
     "Mujeres en Presidencia y vicepresidencias")
surnames = pd.concat([electas[["nombre", "apellido1", "apellido2"]], formulas[["nombre", "apellido1", "apellido2"]]])
surnames = surnames.melt(id_vars="nombre", value_vars=["apellido1", "apellido2"], value_name="Apellido").dropna(subset=["Apellido"])
surname_counts = surnames.groupby("Apellido").nombre.nunique().sort_values(ascending=False).head(20)
display(surname_counts.rename("Nombres completos distintos").to_frame())


,Nombres completos distintos
Apellido,
Vargas,55
Rodríguez,45
Rojas,45
Jiménez,41
Mora,36
González,34
Castro,30
Ramírez,28
Quesada,27


## 15. Evolución del diseño de las papeletas

Se utilizan las 133 observaciones provincia–elección y las características de la base aportada. Sus fuentes son imágenes reconstruidas. Se examinan nombres, suplencias, banderas, disposición, casillas, fondos, encabezados y tipografía; la correlación con voto nulo o blanco no establece un efecto del diseño.

In [20]:
papeletas = frames["papeletas"]
show(px.scatter(papeletas, x="anio", y="provincia", color="Familia de diseño",
                symbol="Familia de diseño", hover_data=["Variante de diseño", "Nombres de candidaturas visibles", "Fondo celeste visible"]),
     "Familias de diseño de papeletas por provincia y elección")
display(papeletas[papeletas.provincia.eq("San José")][["anio", "Familia de diseño", "Variante de diseño", "Nombres de candidaturas visibles", "Posición de casilla", "Fondo celeste visible", "Tipo de fuente"]])


,anio,Familia de diseño,Variante de diseño,Nombres de candidaturas visibles,Posición de casilla,Fondo celeste visible,Tipo de fuente
0,1953,1953 · Columnas iniciales,Encabezado horizontal con serifas,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
7,1958,1958–1994 · Columnas nominales,Reconstrucción de columnas con suplentes,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
14,1962,1958–1994 · Columnas nominales,Lista sin sección de suplencia visible,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
21,1966,1958–1994 · Columnas nominales,Columnas con sustitutos,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
28,1970,1958–1994 · Columnas nominales,Columnas con sustitutos,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
35,1974,1958–1994 · Columnas nominales,Columnas con sustitutos,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
42,1978,1958–1994 · Columnas nominales,Columnas con sustitutos,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
49,1982,1958–1994 · Columnas nominales,Provincia bajo título central,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
56,1986,1958–1994 · Columnas nominales,Provincia bajo título central,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
63,1990,1958–1994 · Columnas nominales,Provincia bajo título central,Sí,Debajo de los nombres,No,Imagen reconstruida aportada


## 16. Cancelaciones y reutilización de nombres

La última cancelación del listado aportado es de junio de 2022. La coincidencia temporal y nominal no verifica continuidad ideológica u organizativa. El HTML incluye también la tabla de coincidencias electorales del archivo original de cancelaciones.

In [21]:
cancelados = frames["cancelados"]
repetidos = cancelados[cancelados["Nombre repetido"].eq("Sí")].groupby("Nombre del partido").size().sort_values()
show(px.bar(x=repetidos.values, y=repetidos.index, orientation="h",
             labels={"x": "Registros", "y": "Denominación partidaria"}),
     "Nombres repetidos en registros de cancelación")
display(cancelados[["Nombre del partido", "Escala", "Provincia", "Fecha de inscripción", "Fecha de cancelación", "Nombre repetido"]].head(20))
display(frames["coincidencias_registrales"].head(10))


,Nombre del partido,Escala,Provincia,Fecha de inscripción,Fecha de cancelación,Nombre repetido
0,Republicano Nacional,Nacional,None,1947-11-07,1952-01-19,Sí
1,Compactación Nacional,Nacional,None,1949-08-13,1952-05-13,No
2,Unión Nacional,Nacional,None,1947-11-04,1952-10-17,Sí
3,Progresista Independiente,Nacional,None,1952-10-25,1956-10-25,No
4,Demócrata,Nacional,None,1952-10-25,1956-10-26,Sí
5,Republicano Nacional Independiente,Nacional,None,1953-03-20,1957-03-20,No
6,Movimiento Democrático Oposicionista,Nacional,None,1956-06-28,1960-06-28,No
7,Demócrata Auténtico,Nacional,None,1957-01-12,1961-01-12,No
8,Independiente,Nacional,None,1957-01-23,1961-01-23,No
9,Republicano Nacional,Nacional,None,1961-01-30,1965-01-30,Sí


,ID registro,Nombre del partido,Nombre comparado,Tipo de denominación,Año electoral,Provincia,Relación temporal,Bases con coincidencia,Nombres en bases electorales
0,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Alajuela,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
1,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Cartago,Posterior al intervalo,Candidaturas legislativas; Resultados legislat...,REPUBLICANO NACIONAL
2,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Guanacaste,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
3,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Heredia,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
4,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Limón,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
5,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Puntarenas,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
6,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,San José,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
7,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1962,Nacional,Posterior al intervalo,Candidaturas presidenciales,Partido Republicano Nacional (PRN)
8,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1974,Alajuela,Posterior al intervalo,Candidaturas legislativas; Resultados legislat...,REPUBLICANO NACIONAL
9,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1974,Cartago,Posterior al intervalo,Candidaturas legislativas; Resultados legislat...,REPUBLICANO NACIONAL


## 17. Laboratorio de asociaciones exploratorias

Las correlaciones se calculan con pares completos. No se seleccionan por significancia ni se interpretan causalmente. Las observaciones tienen dependencia temporal y territorial. El HTML permite cambiar las variables, el año y la provincia y descargar los pares usados.

In [22]:
asociaciones = frames["asociaciones"]
display(asociaciones[asociaciones.ambito.eq("Nacional")].head(15))
show(px.scatter(provincial, x="nep", y="abstencion_pct", color="provincia", hover_data=["anio"],
                labels={"nep": "Número efectivo de partidos", "abstencion_pct": "Abstencionismo (%)"}),
     "Fragmentación y abstencionismo: observaciones provincia–elección")


,ambito,variable_x,variable_y,observaciones,pearson,spearman,interpretacion
0,Nacional,abstencion_pct,invalidos_pct,19,-0.063098,-0.384211,Exploratoria; dependencia temporal y territori...
1,Nacional,abstencion_pct,nep,19,0.700599,0.659649,Exploratoria; dependencia temporal y territori...
2,Nacional,abstencion_pct,concentracion_2,19,-0.785468,-0.798246,Exploratoria; dependencia temporal y territori...
3,Nacional,abstencion_pct,margen,19,0.156634,0.100000,Exploratoria; dependencia temporal y territori...
4,Nacional,abstencion_pct,gallagher,19,0.548572,0.449123,Exploratoria; dependencia temporal y territori...
5,Nacional,abstencion_pct,mujeres_electas_pct,19,0.678212,0.413370,Exploratoria; dependencia temporal y territori...
6,Nacional,abstencion_pct,mujeres_candidatas_pct,19,0.631807,0.384211,Exploratoria; dependencia temporal y territori...
7,Nacional,abstencion_pct,divergencia_arenas,19,0.746428,0.756140,Exploratoria; dependencia temporal y territori...
8,Nacional,abstencion_pct,volatilidad_nominal,18,0.265502,0.329205,Exploratoria; dependencia temporal y territori...
9,Nacional,invalidos_pct,nep,19,-0.474615,-0.729825,Exploratoria; dependencia temporal y territori...


## Ampliación 1.1 · Lectura visual y congruencia

Se conservan los módulos y todas las descargas. La actualización añade rankings provinciales, diferencias pequeñas entre arenas, composición completa por sexo, movilidad provincial con nombres, coincidencia entre ganadores y primera bancada, y secuencias registrales de nombres reutilizados. Las variantes `Unidad Socialcristiana` y `Org Socialista de los Trabajadores` se unifican con sus denominaciones desarrolladas antes de calcular las métricas. Los nombres personales se presentan con capitalización uniforme, conservando `nombre_original`. No se inventan tildes o edades.

Los colores documentados se extraen de las divisas del TSE. Son tonos representativos para visualización, no códigos HEX oficiales ni verificación de continuidad histórica. Las referencias y banderas están en `recursos/colores/` y `recursos/colores_partidos.json`; el HTML muestra las fuentes en Metodología. Para nombres sin divisa documentada se utiliza una paleta de identificación sin carácter oficial.

In [23]:
gaps = frames["brechas_participacion"]
gap_national = gaps[gaps.provincia.eq("Nacional")]
show(px.bar(gap_national, x="anio", y="diferencia_participacion_pp",
            labels={"diferencia_participacion_pp":"Participación legislativa − presidencial (pp)"}),
     "Diferencias de participación entre arenas: escala de puntos porcentuales")
rankings = frames["rankings_provinciales"]
r = rankings[rankings.arena.eq("Legislativa")]
fig = px.line(r, x="anio", y="rango_participacion", color="provincia", markers=True,
              hover_data=["participacion_pct"], color_discrete_map=PROVINCE_COLORS)
fig.update_yaxes(autorange="reversed", dtick=1, title="Puesto · 1 = mayor participación")
show(fig, "Posición de cada provincia en participación legislativa")


### Presidencia electa, voto legislativo y primera bancada

Se distingue el ganador final de la Presidencia, el partido con más votos legislativos y la fuerza con más escaños. Primera bancada no significa mayoría absoluta ni presidencia del Directorio legislativo.

In [24]:
agreement = frames["concordancia_gobierno"]
display(agreement[["anio","nombre","ganador_presidencial","ganador_voto_legislativo","primera_bancada","escanos","escanos_primera_bancada","coincide_primera_bancada"]])
assert len(agreement) == len(YEARS)
assert agreement.escanos_primera_bancada.ge(agreement.escanos).all()
print("Elecciones con Presidencia y primera bancada diferentes:", int((~agreement.coincide_primera_bancada).sum()))


,anio,nombre,ganador_presidencial,ganador_voto_legislativo,primera_bancada,escanos,escanos_primera_bancada,coincide_primera_bancada
0,1953,José Figueres Ferrer,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,30.0,30.0,True
1,1958,Mario Echandi Jiménez,UNION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,10.0,20.0,False
2,1962,Francisco José Orlich Bolmarcich,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,29.0,29.0,True
3,1966,José Joaquín Trejos Fernández,UNIFICACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,26.0,29.0,False
4,1970,José Figueres Ferrer,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,32.0,32.0,True
5,1974,Daniel Oduber Quirós,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,27.0,27.0,True
6,1978,Rodrigo Carazo Odio,UNIDAD,UNIDAD,UNIDAD,27.0,27.0,True
7,1982,Luis Alberto Monge Álvarez,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,33.0,33.0,True
8,1986,Óscar Arias Sánchez,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,29.0,29.0,True
9,1990,Rafael Ángel Calderón Fournier,UNIDAD SOCIAL CRISTIANA,UNIDAD SOCIAL CRISTIANA,UNIDAD SOCIAL CRISTIANA,29.0,29.0,True


Elecciones con Presidencia y primera bancada diferentes: 5


### Movilidad provincial y composición por sexo

Cada movimiento relaciona dos participaciones observadas de una identidad nominal. El año del movimiento es el de destino. La matriz cuenta transiciones; la tabla permite inspeccionar nombres, años y provincias. La edad no se calcula porque las fuentes no contienen fechas de nacimiento.

In [25]:
moves = frames["movimientos"]
province_moves = moves[moves.dimension.eq("Provincia")]
matrix = pd.crosstab(province_moves.origen, province_moves.destino).reindex(index=PROVINCES, columns=PROVINCES, fill_value=0)
fig = go.Figure(go.Heatmap(x=PROVINCES, y=PROVINCES, z=matrix.values,
    texttemplate="%{z}", colorscale=[[0,"#f1f5f9"],[1,"#007f7a"]],
    hovertemplate="Origen: %{y}<br>Destino: %{x}<br>%{z} cambios<extra></extra>"))
fig.update_layout(xaxis_title="Provincia de destino", yaxis_title="Provincia de origen")
show(fig, "Movimientos entre provincias de candidatura")
assert matrix.to_numpy().sum() == len(province_moves)
display(province_moves[["nombre","sexo","anio_origen","origen","anio","destino"]].head(25))
sex = frames["composicion_sexo"]
sex_owners = sex[sex.provincia.eq("Nacional") & sex.poblacion.eq("Candidaturas legislativas") & sex.categoria.eq("Propietaria")]
show(px.bar(sex_owners, x="anio", y="porcentaje", color="sexo", barmode="stack",
            hover_data=["registros","total_categoria"], color_discrete_map=SEX_COLORS),
     "Mujeres y hombres entre candidaturas propietarias")


,nombre,sexo,anio_origen,origen,anio,destino
0,Gabriela Rios Cascante,Mujer,2014,Guanacaste,2018,San José
4,Iris Zamora Zumbado,Mujer,2010,Alajuela,2018,San José
36,Alvaro Escamilla Gutierrez,Hombre,2006,Guanacaste,2014,San José
43,Mauricio Alvarado Delgadillo,Hombre,1986,San José,2006,Puntarenas
50,Carlos Manuel Hernandez Herrera,Hombre,1966,Cartago,1994,San José
57,German Ramon Meza Sibaja,Hombre,1994,San José,2002,Alajuela
62,Laura Guido Perez,Mujer,2014,San José,2018,Cartago
67,José Rafael Cordero Croceri,Hombre,1974,Cartago,1990,Puntarenas
69,Marcial Aguiluz Orellana,Hombre,1958,Puntarenas,1970,San José
78,Adalberto Fonseca Esquivel,Hombre,1990,San José,2006,Heredia


### Cambios de oferta y secuencias de nombres partidarios

La oferta observada distingue participación presidencial y legislativa y el número de provincias; no deduce cambios de escala jurídica. Las fechas de inscripción y cancelación permiten separar intervalos superpuestos de inscripciones posteriores con igual nombre, sin establecer continuidad ideológica.

In [26]:
display(frames["cambios_oferta"].head(25))
reuse = frames["reutilizacion_nombres"]
display(reuse[["nombre_partido","cancelacion_anterior","inscripcion_posterior","dias_entre_registros","relacion","escala_anterior","escala_posterior"]])
assert (reuse["dias_entre_registros"].gt(0) == reuse["relacion"].eq("Inscripción posterior a cancelación")).all()
print("Identidades nominales; fechas y registros originales conservados.")


,partido,anio_origen,anio,oferta_anterior,oferta_actual,alcance_anterior,alcance_actual,provincias_antes,provincias_despues
0,ACCESIBILIDAD SIN EXCLUSION,2006,2010,Solo legislativa,Presidencial y legislativa,Una provincia,Siete provincias,1,7
1,ALIANZA DEMOCRATA CRISTIANA,2014,2018,Solo legislativa,Presidencial y legislativa,Una provincia,Siete provincias,1,7
2,ALIANZA DEMOCRATICA NACIONALISTA,2006,2018,Presidencial y legislativa,Solo legislativa,Siete provincias,Una provincia,7,1
3,ALIANZA PATRIOTICA,2010,2014,Presidencial y legislativa,Solo legislativa,Siete provincias,Varias provincias,7,2
4,DEMOCRATA,1953,1966,Presidencial y legislativa,Solo legislativa,Siete provincias,Siete provincias,7,7
5,DEMOCRATA,1966,1974,Solo legislativa,Presidencial y legislativa,Siete provincias,Siete provincias,7,7
6,FRENTE AMPLIO,2006,2010,Solo legislativa,Presidencial y legislativa,Una provincia,Siete provincias,1,7
7,INDEPENDIENTE,1958,1974,Presidencial y legislativa,Solo legislativa,Siete provincias,Siete provincias,7,7
8,INDEPENDIENTE,1974,1978,Solo legislativa,Presidencial y legislativa,Siete provincias,Siete provincias,7,7
9,INDEPENDIENTE,1978,1982,Presidencial y legislativa,Presidencial y legislativa,Siete provincias,Varias provincias,7,6


,nombre_partido,cancelacion_anterior,inscripcion_posterior,dias_entre_registros,relacion,escala_anterior,escala_posterior
0,Demócrata,1956-10-26,1965-03-08,3055,Inscripción posterior a cancelación,Nacional,Nacional
1,Demócrata,1984-02-01,1997-07-23,4921,Inscripción posterior a cancelación,Nacional,Nacional
2,Nuevo Partido Socialista,2014-03-11,2017-06-14,1191,Inscripción posterior a cancelación,Provincial,Provincial
3,Nuevo Partido Socialista,2018-03-06,2021-07-28,1240,Inscripción posterior a cancelación,Provincial,Provincial
4,Nuevo Partido Socialista,2022-06-08,2021-07-28,-315,Intervalos superpuestos,Provincial,Provincial
5,Republicano Nacional,1952-01-19,1953-03-20,426,Inscripción posterior a cancelación,Nacional,Provincial
6,Republicano Nacional,1957-03-20,1956-10-05,-166,Intervalos superpuestos,Provincial,Provincial
7,Republicano Nacional,1960-10-05,1956-10-30,-1436,Intervalos superpuestos,Provincial,Provincial
8,Republicano Nacional,1960-10-30,1961-01-30,92,Inscripción posterior a cancelación,Provincial,Nacional
9,Republicano Nacional,1965-01-30,1971-06-30,2342,Inscripción posterior a cancelación,Nacional,Nacional


Identidades nominales; fechas y registros originales conservados.


## Ampliación 1.2 · Mapas, perfiles y representación territorial
Autoría: **Juan Ignacio Garbanzo Fallas — politólogo y científico de datos**.
La cartografía provincial se incorpora al HTML sin servicios externos. Perfiles distingue la denominación electoral de los registros jurídicos, sin asignar continuidad no documentada. El laboratorio añade cruces categóricos y el ranking de apellidos muestra todas las etiquetas.
La representación territorial usa el padrón como denominador; no equivale a población censal.

In [27]:
territorial = frames["representacion_territorial"]
assert len(territorial) == 133
assert territorial.groupby("anio").magnitud.sum().tolist() == [45,45]+[57]*17
assert len(data["geografia"]["features"]) == 7
show(px.line(territorial, x="anio", y="magnitud", color="provincia", markers=True, line_shape="hv", labels={"magnitud":"Escaños", "anio":"Elección"}), "Distribución provincial de escaños")
show(px.line(territorial, x="anio", y="indice_representacion", color="provincia", markers=True), "Peso en escaños / peso en padrón")
display(territorial[["anio","provincia","magnitud","cambio_escanos","electores_por_escano","brecha_representacion_pp"]])


,anio,provincia,magnitud,cambio_escanos,electores_por_escano,brecha_representacion_pp
0,1953,Alajuela,8.0,NaN,6916.875000,-1.064287
1,1953,Cartago,6.0,NaN,5829.833333,1.422669
2,1953,Guanacaste,5.0,NaN,5336.400000,2.025650
3,1953,Heredia,3.0,NaN,6552.333333,-0.026719
4,1953,Limón,2.0,NaN,5643.000000,0.601460
...,...,...,...,...,...,...
128,2026,Guanacaste,5.0,1.0,55702.400000,1.171703
129,2026,Heredia,5.0,-1.0,75558.200000,-1.537491
130,2026,Limón,5.0,0.0,62737.000000,0.211878
131,2026,Puntarenas,6.0,1.0,58503.833333,0.947358


## 18. Base final y auditoría de resultados · versión 2.0

El resultado de cada candidatura se toma de su campo `Electa`; la lista separada de electas sirve de cotejo sin desmarcar personas por abreviaturas nominales. La unidad de los porcentajes se declara en cada tabla: **candidatura-evento** o **persona nominal**. Una derrota en una candidatura no se interpreta como el resultado final de toda la carrera: se distingue si la persona había sido electa antes, resultó electa después o nunca registra una elección exitosa en el corpus.

La temporalidad se calcula por **año electoral**. Cuando una persona aparece en dos candidaturas en el mismo año, ninguna se considera antecedente temporal de la otra. Las cédulas y las identidades históricas incompletas impiden equiparar automáticamente nombre con persona civil.

In [28]:
from IPython.display import display

# Autorreparación 1.7: si el kernel conserva `frames` de una integración anterior,
# reconstruir las tablas de outcomes de la versión actual antes de analizarlas.
_required_outcomes = {
    "resultados_candidaturas", "resultados_personas", "transiciones_resultados",
    "puentes_cargos", "antecedentes_cargos_electos", "camara_electa", "ejecutivo_electo"
}
_missing_outcomes = sorted(_required_outcomes.difference(frames))
if _missing_outcomes:
    print("Reconstruyendo tablas de outcomes ausentes del kernel:", ", ".join(_missing_outcomes))
    from atlas_outcomes import build_outcomes
    from atlas_pipeline import norm, records

    _events = frames["eventos"].copy()
    _electas = frames["electas"].copy()
    _formulas = frames["formulas"].copy()
    for _df in (_events, _electas, _formulas):
        _df["nombre_key"] = _df["nombre"].map(norm)

    _rebuilt = build_outcomes(_events, _electas, _formulas)
    for _key, _df in _rebuilt.items():
        frames[_key] = _df.copy()
        data[_key] = records(_df)
        _df.to_csv(OUTPUT_DIR / "datos" / f"{_key}.csv", index=False, encoding="utf-8-sig")

    _still_missing = sorted(_required_outcomes.difference(frames))
    if _still_missing:
        raise RuntimeError("No se pudieron reconstruir las tablas de outcomes: " + ", ".join(_still_missing))
    print("Outcomes reconstruidos correctamente para esta sesión.")

outcome = frames["resultados_candidaturas"]
people_outcome = frames["resultados_personas"]
transitions = frames["transiciones_resultados"]
bridges = frames["puentes_cargos"]
antecedents = frames["antecedentes_cargos_electos"]
chamber = frames["camara_electa"]

assert len(chamber) == 1059
assert chamber.groupby("anio").size().sort_index().tolist() == [45, 45] + [57] * 17
assert int(frames["candidaturas"].electa.sum()) == 1059
assert len(frames["candidaturas"]) + data["meta"]["filas_sin_nombre"] == data["meta"]["filas_fuentes"]["cand"]
assert len(data["geografia"]["features"]) == 7
assert (bridges.anio_origen < bridges.anio).all()
assert (bridges.anio_origen < bridges.anio).all()  # La repetición de cargo es válida si el año posterior es mayor.
print('Puentes al mismo cargo en elecciones posteriores:', int((bridges.cargo_origen == bridges.cargo_destino).sum()))
assert len(antecedents) == int(outcome.electa.sum())
assert set(antecedents.antecedente).issubset({"Ya había resultado electa antes", "Había competido antes, pero aún no había sido electa", "No tenía candidaturas anteriores en estas bases"})

print("Candidaturas en fuente:", len(frames["candidaturas"]) + data["meta"]["filas_sin_nombre"])
print("Candidaturas con nombre:", len(frames["candidaturas"]), "| diputaciones electas:", len(chamber))
print("Personas nominales:", len(people_outcome), "| transiciones consecutivas:", len(transitions))
print("Puentes entre cargos:", len(bridges), "| antecedentes de outcomes electos:", len(antecedents))

rates = []
for label, subset in [("Todas", outcome), ("Primer año documentado", outcome[outcome.primera_candidatura]),
                      ("Año electoral posterior", outcome[outcome.regresa]),
                      ("Ya electa anteriormente", outcome[outcome.electa_antes])]:
    n = int(subset.electa.sum()); den = len(subset)
    rates.append({"Grupo": label, "Electas": n, "Candidaturas": den, "Porcentaje": 100*n/den if den else None})
display(pd.DataFrame(rates))

# Ejemplo reproducible del nuevo puente Diputación -> Presidencia.
bridge_example = bridges[(bridges.cargo_origen == "Diputación") & (bridges.cargo_destino == "Presidencia")]
bridge_people = bridge_example.groupby("id_persona").agg(nombre=("nombre", "last"),
    pares=("id_destino", "size"), destino_electo=("electa_destino", "max")).reset_index()
display(bridge_people.sort_values(["destino_electo", "pares"], ascending=False).head(20))

show(px.bar(pd.DataFrame(rates), x="Grupo", y="Porcentaje", text="Electas",
            labels={"Porcentaje":"Candidaturas electas (%)", "Grupo":"Condición previa"}),
     "Éxito electoral descriptivo por experiencia")
print("Las asociaciones son descriptivas. No atribuir causalidad, identidad civil o parentesco a coincidencias nominales.")

Puentes al mismo cargo en elecciones posteriores: 2611
Candidaturas en fuente: 14874
Candidaturas con nombre: 14870 | diputaciones electas: 1059
Personas nominales: 13183 | transiciones consecutivas: 2191
Puentes entre cargos: 2977 | antecedentes de outcomes electos: 1116


,Grupo,Electas,Candidaturas,Porcentaje
0,Todas,1116,15400,7.246753
1,Primer año documentado,790,13249,5.962714
2,Año electoral posterior,326,2151,15.155742
3,Ya electa anteriormente,150,348,43.103448


,id_persona,nombre,pares,destino_electo
16,55b570dd15b317,Luis Alberto Monge Álvarez,4,True
19,5ed5c12bb01e4e,Rodrigo Carazo Odio,4,True
6,14bf391b934645,Rafael Ángel Calderón Fournier,3,True
37,b33964bddee8d1,Mario Echandi Jiménez,3,True
46,dbe07284e6ddb3,Daniel Oduber Quirós,3,True
4,11bf4727a7d350,Francisco José Orlich Bolmarcich,2,True
14,45488e796aaf7c,Óscar Arias Sánchez,2,True
15,4fac866f683a09,Abel Pacheco de la Espriella,2,True
29,8640f03f7ceb18,Miguel Ángel Rodríguez Echeverría,2,True
24,7de24a499c9b84,Laura Chinchilla Miranda,1,True


Las asociaciones son descriptivas. No atribuir causalidad, identidad civil o parentesco a coincidencias nominales.


### 18.1 Auditoría adicional de claridad: candidatura, persona y carrera completa

Estas comprobaciones sostienen las vistas de **Éxito electoral**, **Explorador de personas**, **Sexo y representación** y **Fórmulas presidenciales**. Se verifican denominadores, el contexto temporal de candidaturas no electas, la recurrencia por años electorales y los porcentajes de elección por cargo y sexo. Las asociaciones son descriptivas.

In [29]:
# Auditoría vectorizada: la condición "electa antes/después" depende de años
# estrictamente anteriores/posteriores, nunca del orden de candidaturas simultáneas.
_year = (outcome.groupby(["id_persona","anio"], as_index=False)
         .agg(electa_anio=("electa","max"))
         .sort_values(["id_persona","anio"]))
_prior_cum = _year.groupby("id_persona")["electa_anio"].cummax()
_year["prior_expected"] = _prior_cum.groupby(_year["id_persona"]).shift(fill_value=False).astype(bool)
_rev = _year.iloc[::-1].copy()
_later_cum = _rev.groupby("id_persona")["electa_anio"].cummax()
_rev["later_expected"] = _later_cum.groupby(_rev["id_persona"]).shift(fill_value=False).astype(bool)
_year["later_expected"] = _rev["later_expected"].iloc[::-1].to_numpy()
_audit = outcome.merge(_year[["id_persona","anio","prior_expected","later_expected"]],
                       on=["id_persona","anio"], how="left", validate="many_to_one")
assert _audit.electa_antes.eq(_audit.prior_expected).all()
assert _audit.electa_despues.eq(_audit.later_expected).all()

# Resumen directo por cargo: candidaturas, electas, no electas y tasa.
by_role = (outcome.groupby("cargo")
           .agg(candidaturas=("id_evento","size"), electas=("electa","sum"))
           .reset_index())
by_role["no_electas"] = by_role.candidaturas - by_role.electas
by_role["tasa_eleccion_pct"] = 100 * by_role.electas / by_role.candidaturas
display(by_role.sort_values("candidaturas", ascending=False))

# Resumen por sexo dentro de cada cargo, con denominador explícito.
by_role_sex = (outcome[outcome.sexo.isin(["Mujer","Hombre"])]
               .groupby(["cargo","sexo"])
               .agg(candidaturas=("id_evento","size"), electas=("electa","sum"))
               .reset_index())
by_role_sex["no_electas"] = by_role_sex.candidaturas - by_role_sex.electas
by_role_sex["tasa_eleccion_pct"] = 100 * by_role_sex.electas / by_role_sex.candidaturas
display(by_role_sex)

# Las candidaturas no electas se clasifican por lo que ocurre en la carrera completa.
losses = outcome[~outcome.electa].copy()
losses["contexto_carrera"] = np.select(
    [losses.electa_despues, (~losses.electa_despues) & losses.electa_antes],
    ["Resultó electa después", "Ya había resultado electa antes"],
    default="Nunca resultó electa en el corpus"
)
display(losses.groupby("contexto_carrera").agg(candidaturas=("id_evento","size"), personas=("id_persona","nunique")).reset_index())

# Rutas completas de personas hasta su primera elección exitosa.
display(people_outcome.groupby("ruta_primera_eleccion").size().sort_values(ascending=False).rename("personas").to_frame())
print("Auditoría 1.7 completada: temporalidad por año electoral y denominadores explícitos.")

,cargo,candidaturas,electas,no_electas,tasa_eleccion_pct
0,Diputación,14870,1059,13811,7.121722
1,Presidencia,178,19,159,10.674157
2,Primera Vicepresidencia,176,19,157,10.795455
3,Segunda Vicepresidencia,176,19,157,10.795455


,cargo,sexo,candidaturas,electas,no_electas,tasa_eleccion_pct
0,Diputación,Hombre,10296,833,9463,8.090521
1,Diputación,Mujer,4572,226,4346,4.943132
2,Presidencia,Hombre,163,17,146,10.429448
3,Presidencia,Mujer,15,2,13,13.333333
4,Primera Vicepresidencia,Hombre,97,15,82,15.463918
5,Primera Vicepresidencia,Mujer,79,4,75,5.063291
6,Segunda Vicepresidencia,Hombre,110,14,96,12.727273
7,Segunda Vicepresidencia,Mujer,66,5,61,7.575758


,contexto_carrera,candidaturas,personas
0,Nunca resultó electa en el corpus,13867,12223
1,Resultó electa después,250,195
2,Ya había resultado electa antes,167,126


,personas
ruta_primera_eleccion,
Nunca resultó electa en el corpus,12217
Resultó electa en su primer año documentado,790
Resultó electa después de 1 elección previa sin éxito,143
Resultó electa después de 2 elecciones previas sin éxito,30
Resultó electa después de 3 elecciones previas sin éxito,3


Auditoría 1.7 completada: temporalidad por año electoral y denominadores explícitos.


## 19. Aplicación interactiva · versión 2.0

El HTML contiene comparación provincial y temporal, composición de la cámara por elección, perfiles nominales, coincidencias de apellidos, resultados por candidatura y persona, transiciones, puentes entre cargos, representación territorial y un centro de descarga de datos. Incluye la biblioteca gráfica, los datos y la geometría provincial dentro del mismo archivo, por lo que puede abrirse sin conexión.


## 20. Generación de la aplicación HTML · versión 2.0

El resultado conserva los módulos, filtros de año y territorio, controles específicos por módulo, búsqueda y ordenamiento de tablas, trayectorias individuales, explorador de datos y laboratorio de relaciones. Cada gráfico ofrece PNG, SVG y CSV; las tablas descargan todas las filas de la búsqueda. El Explorador de datos permite además descargar paquetes temáticos de varios CSV o una selección personalizada en JSON. Las vistas pueden guardarse y recargarse como JSON. Plotly y los datos están dentro del HTML: no necesita servidor ni conexión.

La autoría y el contexto IFED–TSE identifican el producto de la Práctica Profesional. Es un entregable interactivo para consulta y análisis y no sustituye las publicaciones oficiales del Tribunal.


### Archivos de salida del Atlas público

Al ejecutar este notebook se actualizan **solo** `salidas/datos/` y `salidas/Atlas_Electoral_Costa_Rica_1953_2026.html`. Las figuras para los capítulos son un producto **independiente**: no se crean carpetas de gráficos editoriales ni un segundo HTML. No borrar los datos originales ni los recursos del proyecto.


In [30]:
# V14 · Conservación de recursos y de la aplicación pública.
# No reinstalar extensiones previas ni sobrescribir JavaScript en cada ejecución.
assert (PROJECT_DIR/'recursos'/'atlas_final.js').is_file()
assert (PROJECT_DIR/'recursos'/'atlas_v12.js').is_file()
assert (PROJECT_DIR/'recursos'/'atlas_v14.js').is_file()
assert (PROJECT_DIR/'recursos'/'atlas_template.html').is_file()
print('Módulos del Atlas V16 verificados; la arquitectura estable se conserva sin parches destructivos.')


Módulos del Atlas V16 verificados; la arquitectura estable se conserva sin parches destructivos.


## 21. V20 · Movimientos y títulos auditables
Dos diagramas independientes en Personas y trayectorias: cargos y provincias, con selector de población para la vista provincial. El módulo Éxito electoral permite Todos en cargo anterior y posterior y contextualiza las tasas en título, subtítulo y PNG. Sin exportación de figuras históricas por separado.


In [31]:
assert (PROJECT_DIR/'recursos'/'atlas_final.js').is_file()
assert (PROJECT_DIR/'recursos'/'atlas_v14.js').is_file()
data['meta']['version']='V22.0 · archivo presidencial, hemiciclo y datos accesibles'
print('V22.0: módulos verificados; métricas y visualizaciones preservadas; sin PNG externos.')


V22.0: módulos verificados; métricas y visualizaciones preservadas; sin PNG externos.


In [32]:
# V21.2 · Ajustes puntuales y reversibles durante la generación del HTML.
# 1. Añade porcentajes VISIBLES en los nodos de TODOS los Sankey.
# 2. Mantiene también la lectura por cursor dentro de cada conexión.
# 3. Añade a las tres poblaciones presidenciales los hitos históricos de género.
# El paquete y los recursos originales quedan intactos. No crea PNG externos.
from tempfile import TemporaryDirectory
from pathlib import Path
import shutil
from atlas_pipeline import render_app

ATLAS_V21_AJUSTES_JS = r'''/* ATLAS V21.2: porcentajes visibles en Sankey + referencias visuales de género.
 * Añadido al HTML durante la ejecución del notebook sin modificar recursos/.
 */
(function atlasV21DosCambiosVisibles(){
  if(typeof plot!=='function') throw new Error('Falta la función de gráficos del Atlas.');
  const plotAnterior=plot;
  const porcentaje=(n,total)=>total>0?(100*n/total).toLocaleString('es-CR',{minimumFractionDigits:1,maximumFractionDigits:1})+' %':'—';
  const limpiarEtiqueta=texto=>String(texto)
    .replace(/\s*[·|]\s*[0-9.,]+\s*%\s*(del total)?\s*$/i,'')
    .replace(/\s*\([^)]*%[^)]*\)\s*$/,'')
    .trim();
  const hitos=[
    {year:1998,label:'Cuota 40 %'},
    {year:2002,label:'Puestos elegibles'},
    {year:2014,label:'Paridad vertical'},
    {year:2018,label:'Paridad horizontal'}
  ];
  plot=function(titulo,trazas,filas,opciones={}){
    let actualizadas=trazas;
    let configuracion={...opciones};
    if(Array.isArray(trazas)&&trazas.some(t=>t?.type==='sankey')){
      actualizadas=trazas.map(t=>{
        if(t?.type!=='sankey') return t;
        const nodos=t.node||{}, enlaces=t.link||{};
        const etiquetas=Array.isArray(nodos.label)?nodos.label:[];
        const origen=Array.isArray(enlaces.source)?enlaces.source:[];
        const destino=Array.isArray(enlaces.target)?enlaces.target:[];
        const pesos=Array.isArray(enlaces.value)?enlaces.value:[];
        if(!etiquetas.length||!pesos.length) return t;
        const salidas=etiquetas.map(()=>0), entradas=etiquetas.map(()=>0);
        for(let i=0;i<pesos.length;i++){
          const s=Number(origen[i]), d=Number(destino[i]), v=Number(pesos[i]);
          if(!Number.isInteger(s)||!Number.isInteger(d)||s<0||d<0||s>=etiquetas.length||d>=etiquetas.length||!Number.isFinite(v)||v<0) continue;
          salidas[s]+=v;
          entradas[d]+=v;
        }
        const total=salidas.reduce((a,b)=>a+b,0);
        if(total<=0) return t;
        const nombres=etiquetas.map((texto,i)=>{
          const base=limpiarEtiqueta(texto);
          if(salidas[i]>0) return `${base} · ${porcentaje(salidas[i],total)}`;
          if(entradas[i]>0) return `${base} · ${porcentaje(entradas[i],total)}`;
          return base;
        });
        const hoverOriginal=enlaces.hovertemplate;
        const hovers=pesos.map((v,i)=>{
          const s=Number(origen[i]), d=Number(destino[i]);
          const base=Array.isArray(hoverOriginal)?hoverOriginal[i]:hoverOriginal;
          const original=(typeof base==='string'&&base)?base:'%{value}<extra></extra>';
          const anexo=`<br>Participación dentro del origen: ${porcentaje(Number(v),salidas[s]||0)}`+
            `<br>Participación dentro del destino: ${porcentaje(Number(v),entradas[d]||0)}`;
          return original.includes('<extra>') ? original.replace('<extra>',anexo+'<extra>') : original+anexo+'<extra></extra>';
        });
        return {
          ...t,
          node:{...nodos,label:nombres},
          link:{...enlaces,hovertemplate:hovers}
        };
      });
      configuracion={
        ...configuracion,
        note:[
          configuracion.note,
          'Los porcentajes visibles junto a cada nodo representan la proporción del peso total de las conexiones mostradas. En el cursor de cada enlace también se informa qué proporción representa dentro de su origen y dentro de su destino. Se conservan los conteos y filtros originales; un enlace no equivale necesariamente a una persona única.'
        ].filter(Boolean).join(' ')
      };
    }
    const poblacion=String(typeof local!=='undefined'?(local.sexpop||''):'');
    const esGeneroTemporal=String(titulo).startsWith('Composición por sexo a través del tiempo');
    if(esGeneroTemporal&&['Presidencia','Primera Vicepresidencia','Segunda Vicepresidencia'].includes(poblacion)){
      const anios=new Set((Array.isArray(actualizadas)?actualizadas:[]).flatMap(t=>Array.isArray(t.x)?t.x.map(Number):[]));
      const visibles=hitos.filter(h=>anios.has(h.year));
      const distribucion=[1.09,1.19,1.09,1.19];
      if(visibles.length){
        const layout=configuracion.layout||{};
        const formaNueva=visibles.map(h=>({type:'line',xref:'x',yref:'paper',x0:h.year,x1:h.year,y0:0,y1:1,line:{color:'#b28744',width:1.7,dash:'dot'}}));
        const notaNueva=visibles.map((h,i)=>({xref:'x',yref:'paper',x:h.year,y:distribucion[i],text:`${h.year} · ${h.label}`,showarrow:false,xanchor:'center',font:{size:10,color:'#755c32'},bgcolor:'rgba(255,250,241,.98)',bordercolor:'rgba(178,135,68,.4)',borderpad:3}));
        configuracion={
          ...configuracion,
          height:Math.max(560,configuracion.height||0),
          layout:{...layout,shapes:[...(layout.shapes||[]),...formaNueva],annotations:[...(layout.annotations||[]),...notaNueva],margin:{...(layout.margin||{}),t:175}},
          note:[configuracion.note,'Hitos de referencia de la normativa de género en elecciones costarricenses (1998, 2002, 2014, 2018). Las etiquetas se muestran como contexto histórico comparativo: las reglas legislativas de cuotas, puestos elegibles y paridad no deben suponerse automáticamente aplicables a las candidaturas presidenciales o vicepresidenciales. La coincidencia temporal no establece causalidad.'].filter(Boolean).join(' ')
        };
      }
    }
    return plotAnterior(titulo,actualizadas,filas,configuracion);
  };
})();
'''
# V21.3 · Módulo autónomo añadido durante render, sin tocar recursos/.
ATLAS_V21_AJUSTES_JS += r'''
/* ATLAS_V21_4_DOBLE_POSTULACION_CLARIDAD
 * Extensión autónoma que no altera bases, otros módulos ni recursos permanentes.
 * Doble postulación: misma identidad nominal + mismo año de elección,
 * Presidencia y Diputación propietaria simultáneas. Fórmulas vicepresidenciales
 * se excluyen de esta definición, y Nacional no se trata como provincia.
 */
(function atlasV213Dobles(){
 'use strict';
 if(typeof views==='undefined'||typeof plot!=='function'||typeof D==='undefined'||!Array.isArray(D.eventos)) return;
 const nation='Costa Rica';
 const fmt=(n,d=0)=>Number(n).toLocaleString('es-CR',{maximumFractionDigits:d,minimumFractionDigits:d});
 let cached=null;
 function observations(){
  if(cached)return cached;
  const events=D.eventos.filter(r=>r&&r.id_persona&&Number.isInteger(Number(r.anio)));
  const by=new Map();
  for(const ev of events){const key=String(ev.id_persona)+'|'+String(ev.anio);if(!by.has(key))by.set(key,[]);by.get(key).push(ev)}
  const out=[];
  for(const [key,list] of by){
   const presidential=list.filter(r=>r.cargo==='Presidencia');
   const legislative=list.filter(r=>r.cargo==='Diputación'&&r.tipo==='Propietaria'&&P.includes(r.provincia));
   if(!presidential.length||!legislative.length)continue;
   const provinces=uniq(legislative.map(r=>r.provincia));
   const parties=uniq([...presidential,...legislative].map(r=>r.partido));
   // En los archivos verificados existe una sola candidatura a cada cargo.
   // Los casos de ambigüedad se conservan para auditoría, sin multiplicar personas.
   const firstP=presidential[0],firstL=legislative[0];
   out.push({id_persona:firstP.id_persona,anio:Number(firstP.anio),nombre:firstP.nombre,
    identidad:firstP.identidad,sexo:firstP.sexo,provincia:provinces.length===1?provinces[0]:'Ambigua',
    partido:parties.length===1?parties[0]:'Partidos distintos',
    partido_presidencial:firstP.partido,partido_legislativo:firstL.partido,
    mismo_partido:presidential.some(p=>legislative.some(l=>p.partido===l.partido)),
    electa_presidencia:presidential.some(r=>r.electa===true),electa_diputacion:legislative.some(r=>r.electa===true),
    numero_candidaturas_presidenciales:presidential.length,numero_candidaturas_legislativas:legislative.length,
    id_presidencial:firstP.id_evento,id_legislativo:firstL.id_evento,
    orden_lista:firstL.orden,ambiguo:provinces.length!==1||presidential.length!==1||legislative.length!==1});
  }
  out.sort((a,b)=>a.anio-b.anio||a.nombre.localeCompare(b.nombre,'es'));
  const prior=new Map();
  for(const x of out){const earlier=prior.get(x.id_persona);
   x.repeticion_doble=Boolean(earlier);
   x.anio_doble_anterior=earlier?earlier.anio:null;
   x.continuidad_partido=earlier?(earlier.partido===x.partido?'Mantiene partido':'Cambia partido'):'Primera doble postulación';
   x.continuidad_provincia=earlier?(earlier.provincia===x.provincia?'Mantiene provincia':'Cambia provincia'):'Primera doble postulación';
   prior.set(x.id_persona,x);
  }
  cached=out;
  return out;
 }
 function draw(){
  add('<div class="extension-heading"><span>DOBLE POSTULACIÓN · PRESIDENCIA Y DIPUTACIÓN</span></div>');
  add('<div class="clarity-box"><b>Doble postulación:</b> la misma persona compite simultáneamente a la <b>Presidencia</b> y por una <b>diputación propietaria</b> en una misma elección. <b>Reiteración de doble postulación:</b> un nuevo episodio en otra elección posterior de alguien que ya había presentado ambas candidaturas simultáneamente. <b>No es reelección ni mera repetición de una candidatura individual.</b> Cada caso persona–elección se cuenta una sola vez; una persona puede aportar varios casos. Vicepresidencias no se incluyen por sí solas. La provincia se refiere solo a la candidatura legislativa; la presidencial es nacional. Las identidades nominales y resultados requieren revisión documental. No se infiere causalidad.</div>');
  const all=observations();
  const provinces=['Todas las provincias',...P];
  const electionResult=[['Todos','Todos los resultados'],['Diputación electa','Diputación electa'],['Ninguna electa','Ninguna candidatura electa'],['Presidencia electa','Presidencia electa'],['Ambos cargos electos','Ambos cargos electos']];
  const repeatOptions=[['Todos','Todas las dobles postulaciones'],['Primera','Primera doble postulación de la persona'],['Repetida','Persona que ya había tenido doble postulación en otra elección']];
  const continuity=['Todas','Primera doble postulación','Mantiene partido','Cambia partido'];
  const provincesCont=['Todas','Primera doble postulación','Mantiene provincia','Cambia provincia'];
  controls([
   ['dobProv','Provincia de la candidatura a diputación',provinces,local.dobProv||provinces[0]],
   ['dobOutcome','Resultado combinado',electionResult,local.dobOutcome||'Todos'],
   ['dobRepeat','¿Volvió a postularse simultáneamente a los dos cargos?',repeatOptions,local.dobRepeat||'Todos'],
   ['dobPartyChange','Partido entre dobles postulaciones',continuity,local.dobPartyChange||'Todas'],
   ['dobProvChange','Provincia entre dobles postulaciones',provincesCont,local.dobProvChange||'Todas']
  ]);
  const date=all.filter(r=>r.anio>=state.from&&r.anio<=state.to);
  const filtered=date.filter(r=>
   (local.dobProv===undefined||local.dobProv==='Todas las provincias'||r.provincia===local.dobProv)&&
   (!local.dobOutcome||local.dobOutcome==='Todos'||
    (local.dobOutcome==='Diputación electa'&&r.electa_diputacion)||
    (local.dobOutcome==='Presidencia electa'&&r.electa_presidencia)||
    (local.dobOutcome==='Ninguna electa'&&!r.electa_diputacion&&!r.electa_presidencia)||
    (local.dobOutcome==='Ambos cargos electos'&&r.electa_diputacion&&r.electa_presidencia))&&
   (!local.dobRepeat||local.dobRepeat==='Todos'||(local.dobRepeat==='Repetida'?r.repeticion_doble:!r.repeticion_doble))&&
   (!local.dobPartyChange||local.dobPartyChange==='Todas'||r.continuidad_partido===local.dobPartyChange)&&
   (!local.dobProvChange||local.dobProvChange==='Todas'||r.continuidad_provincia===local.dobProvChange));
  const people=uniq(filtered.map(r=>r.id_persona)).length;
  const yearFirst=all.length?Math.min(...all.map(r=>r.anio)):null;
  kpis([
   ['Casos persona–elección',fmt(filtered.length),'Cada persona en una elección cuenta una vez'],
   ['Personas nominales distintas',fmt(people),'No equivale a candidatos en años distintos'],
   ['Diputaciones electas',fmt(filtered.filter(r=>r.electa_diputacion).length),'Resultados simultáneos de las dobles postulaciones'],
   ['Presidencias electas',fmt(filtered.filter(r=>r.electa_presidencia).length),'Resultados simultáneos de las dobles postulaciones']
  ]);
  const scope=[local.dobProv&&local.dobProv!=='Todas las provincias'?local.dobProv:'Costa Rica',
   local.dobOutcome&&local.dobOutcome!=='Todos'?local.dobOutcome:'Todos los resultados',
   local.dobRepeat&&local.dobRepeat!=='Todos'?local.dobRepeat:null,
   local.dobPartyChange&&local.dobPartyChange!=='Todas'?local.dobPartyChange:null,
   local.dobProvChange&&local.dobProvChange!=='Todas'?local.dobProvChange:null].filter(Boolean).join(' · ');
  add(`<div class="clarity-box"><b>Primera observación en el corpus completo:</b> ${yearFirst??'sin casos'} · <b>Lectura activa:</b> ${esc(scope)} · ${state.from}–${state.to}. Las cifras se basan en las candidaturas registradas en las bases de esta versión, no en un inventario jurídico exhaustivo de toda la historia electoral.</div>`);
  const ys=yearsVisible();
  const longitudinal=ys.map(anio=>{
   const gg=filtered.filter(r=>r.anio===anio);
   const pYear=uniq(D.eventos.filter(r=>r.anio===anio&&r.cargo==='Presidencia').map(r=>r.id_persona)).length;
   return {anio,postulaciones_dobles:gg.length,personas:gg.length,porcentaje_candidaturas_presidenciales:pYear?100*gg.length/pYear:null,candidaturas_presidenciales:pYear,electas_diputacion:gg.filter(r=>r.electa_diputacion).length,electas_presidencia:gg.filter(r=>r.electa_presidencia).length};
  });
  plot('Doble postulación presidencial–legislativa · evolución histórica',
   [{type:'scatter',mode:'lines+markers',name:'Postulaciones dobles',x:ys,y:longitudinal.map(r=>r.postulaciones_dobles),line:{color:'#087f87',width:3},marker:{size:9},customdata:longitudinal.map(r=>[r.candidaturas_presidenciales,r.porcentaje_candidaturas_presidenciales]),hovertemplate:'%{x}<br>%{y} dobles postulaciones<br>Personas candidatas a Presidencia: %{customdata[0]}<br>Proporción: %{customdata[1]:.1f}%<extra></extra>'},
    {type:'scatter',mode:'lines+markers',name:'Reiteraciones (ambos cargos, en otra elección posterior)',x:ys,y:ys.map(y=>filtered.filter(r=>r.anio===y&&r.repeticion_doble).length),line:{color:'#bc8b5b',width:2,dash:'dash'},marker:{size:7},hovertemplate:'%{x}<br>%{y} episodios nuevos de personas que ya habían tenido una doble postulación<extra></extra>'}],longitudinal,
   {height:430,subtitle:`${scope} · ${state.from}–${state.to}`,yTitle:'Personas con dos candidaturas simultáneas',note:'Primera línea: total de personas que se presentaron simultáneamente a Presidencia y diputación propietaria en cada elección. Segunda línea: entre esas personas, cuántas ya habían realizado una doble postulación en una elección anterior (reiteraciones, no personas nuevas). Un cero indica ausencia de casos identificados en las bases, no prohibición legal. Cambios descriptivos, sin inferencia causal.'});
  const geos=P.map(p=>({provincia:p,registros:filtered.filter(r=>r.provincia===p).length,personas:uniq(filtered.filter(r=>r.provincia===p).map(r=>r.id_persona)).length}));
  const heatRows=geos.flatMap(g=>ys.map(anio=>({provincia:g.provincia,anio,casos:filtered.filter(r=>r.anio===anio&&r.provincia===g.provincia).length})));
  heatRows.push(...ys.map(anio=>({provincia:nation,anio,casos:filtered.filter(r=>r.anio===anio).length})));
  const totals=geos.map(g=>({provincia:g.provincia,total:g.registros}));
  const xYears=ys.map(String),yProvinces=[...P];
  const matrixRows=geos.map(g=>ys.map(anio=>filtered.filter(r=>r.anio===anio&&r.provincia===g.provincia).length));
  const totalByProvince=geos.map(g=>g.registros),totalByYear=ys.map(anio=>filtered.filter(r=>r.anio===anio).length);
  const matrixAudit=[...heatRows,
    ...P.map((provincia,i)=>({provincia,anio:'Total histórico',casos:totalByProvince[i],tipo:'Total por provincia'})),
    ...ys.map((anio,i)=>({provincia:nation,anio,casos:totalByYear[i],tipo:'Total país por elección'})),
    {provincia:nation,anio:'Total histórico',casos:filtered.length,tipo:'Total país y periodo'}];
  if(filtered.length){
   // V21.7: UNA SOLA escala cromática para cuerpo, columna total y fila Costa Rica.
   // 0 es un valor observado; se conserva y se visualiza como 0.
   // ÚNICA excepción: esquina país/periodo (gran total, sin escala).
   const xAll=[...xYears,'Total histórico'];
   const yAll=[...yProvinces,nation];
   const escalaMax=Math.max(1,...matrixRows.flat(),...totalByProvince,...totalByYear);
   // Una sola paleta teal: un mismo valor siempre tiene exactamente el mismo
   // color en toda la matriz. Los puntos de control evitan perder contraste
   // para valores pequeños, sin cambiar los conteos ni los ceros.
   const puntosColor=[
     [0,'#f0f6f7'],[1,'#e4f0f1'],[2,'#cbe4e5'],[4,'#9dccd0'],
     [6,'#70b5bc'],[8,'#449ca6'],[10,'#27818c'],[12,'#186b79'],
     [20,'#125164'],[33,'#0b354d']
   ];
   const colorscale=[];
   for(const [valor,color] of puntosColor){
     if(valor<=escalaMax) colorscale.push([valor/escalaMax,color]);
   }
   if(colorscale[colorscale.length-1][0]<1){
     colorscale.push([1,puntosColor.find(p=>p[0]>escalaMax)?.[1]||'#0b354d']);
   }
   const grid=[
     ...matrixRows.map((fila,i)=>[...fila,totalByProvince[i]]),
     [...totalByYear,null]
   ];
   const texto=grid.map((fila,i)=>fila.map((n,j)=>
     n===null?'':(n>=8?'':String(n))));
   const base={type:'heatmap',x:xAll,y:yAll,z:grid,text:texto,
      texttemplate:'%{text}',textfont:{size:11,color:'#173849'},
      zmin:0,zmax:escalaMax,colorscale,
      xgap:1.5,ygap:1.5,
      colorbar:{title:{text:'Casos'},len:.74,thickness:15,
        tickmode:'array',tickvals:[0,1,2,4,8,12,20,33].filter(v=>v<=escalaMax)},
      hovertemplate:'%{y} · %{x}<br>%{z} casos persona–elección<extra></extra>'};
   const formas=[],etiquetas=[];
   // V21.8: en lugar de anotaciones blancas (que desaparecían al exportar),
   // dibujamos los valores altos como una traza de texto encima del heatmap.
   // El calor y el denominador NO cambian; se conserva la etiqueta "0".
   const textoIntenso={x:[],y:[],text:[]};
   for(let i=0;i<grid.length;i++)for(let j=0;j<grid[i].length;j++){
     const n=grid[i][j];
     if(n!==null && n>=8){
       textoIntenso.x.push(xAll[j]);
       textoIntenso.y.push(yAll[i]);
       textoIntenso.text.push(String(n));
     }
   }
   const capaTextoIntenso={
      type:'scatter',mode:'text',x:textoIntenso.x,y:textoIntenso.y,
      text:textoIntenso.text,textposition:'middle center',
      textfont:{size:11,color:'#ffffff',family:'Arial, sans-serif'},
      hoverinfo:'skip',showlegend:false,cliponaxis:false
   };
   // Solo esta esquina se mantiene completamente BLANCA y aislada.
   formas.push({type:'rect',xref:'x',yref:'y',x0:ys.length-.5,x1:ys.length+.5,
    y0:P.length-.5,y1:P.length+.5,fillcolor:'#ffffff',
    line:{color:'#15558c',width:2.6},layer:'above'});
   etiquetas.push({xref:'x',yref:'y',x:'Total histórico',y:nation,
    text:`<b>${filtered.length}</b>`,showarrow:false,
    font:{size:17,color:'#13578e'},xanchor:'center',yanchor:'middle'});
   // Separadores estructurales: mismos ejes y tamaños de celda.
   formas.push({type:'line',xref:'x',yref:'y',x0:ys.length-.5,x1:ys.length-.5,
    y0:-.5,y1:P.length+.5,line:{color:'#335d77',width:2.1},layer:'above'});
   formas.push({type:'line',xref:'x',yref:'y',x0:-.5,x1:ys.length+.5,
    y0:P.length-.5,y1:P.length-.5,line:{color:'#335d77',width:2.1},layer:'above'});
   plot('Doble postulación por provincia y elección · total Costa Rica',
    [base,capaTextoIntenso],matrixAudit,
    {height:615,subtitle:`${scope} · ${state.from}–${state.to}`,
     xTitle:'Elección',yTitle:'Provincia de la candidatura legislativa',
     layout:{xaxis:{type:'category',tickangle:-45,
       categoryorder:'array',categoryarray:xAll,
       range:[-.55,xAll.length-.45]},
      yaxis:{type:'category',autorange:'reversed',
       categoryorder:'array',categoryarray:yAll,
       range:[yAll.length-.48,-.52]},
      margin:{l:125,r:95,b:115,t:35},
      shapes:formas,annotations:etiquetas},
     note:'El mismo valor de casos utiliza el mismo color en toda la matriz: provincias por elección, totales históricos provinciales y totales nacionales anuales comparten una única escala. Los ceros son valores válidos y se muestran como 0. La casilla blanca de la esquina inferior derecha muestra exclusivamente el total país–periodo (41 en el conjunto completo), sin degradado. Cada celda anual cuenta casos persona–elección de doble postulación a Presidencia y diputación propietaria; una identidad puede aparecer en distintas elecciones.'});
   const ordered=geos.slice().sort((a,b)=>b.registros-a.registros);
   plot('Concentración territorial de las dobles postulaciones',
    [{type:'bar',orientation:'h',x:ordered.map(r=>r.registros),y:ordered.map(r=>r.provincia),text:ordered.map(r=>`${r.registros} · ${fmt(100*r.registros/filtered.length,1)} %`),textposition:'outside',cliponaxis:false,marker:{color:ordered.map(r=>provinceColors[r.provincia]||'#27818b')},hovertemplate:'%{y}<br>%{x} casos persona–elección<extra></extra>'}],ordered,
    {height:410,subtitle:`${scope}`,xTitle:'Dobles postulaciones documentadas',note:'Provincia de la candidatura a diputación; no equivale al lugar de origen o residencia de las personas.'});
   // Tasas por provincia: numerador = candidaturas a diputación electas;
   // denominador = casos persona–elección con doble postulación en la provincia.
   const provinceSuccess=geos.map(g=>{
     const observationsHere=filtered.filter(r=>r.provincia===g.provincia);
     const n=observationsHere.length,won=observationsHere.filter(r=>r.electa_diputacion).length;
     return {provincia:g.provincia,casos:n,electas:won,no_electas:n-won,tasa_eleccion_pct:n?100*won/n:null};
   }).filter(g=>g.casos>0).sort((a,b)=>a.tasa_eleccion_pct-b.tasa_eleccion_pct);
   if(provinceSuccess.length)plot('Éxito legislativo entre las dobles postulaciones · por provincia',
    [{type:'bar',orientation:'h',x:provinceSuccess.map(r=>r.tasa_eleccion_pct),
      y:provinceSuccess.map(r=>r.provincia),text:provinceSuccess.map(r=>`${fmt(r.tasa_eleccion_pct,1)} % · ${r.electas}/${r.casos}`),
      textposition:'outside',cliponaxis:false,marker:{color:provinceSuccess.map(r=>provinceColors[r.provincia]||'#167f83')},
      customdata:provinceSuccess.map(r=>[r.electas,r.casos]),
      hovertemplate:'%{y}<br>%{customdata[0]} diputaciones electas de %{customdata[1]} dobles postulaciones · %{x:.1f}%<extra></extra>'}],provinceSuccess,
    {height:420,subtitle:`${scope}`,xTitle:'Porcentaje que resultó electo a diputación (%)',layout:{margin:{l:130,r:120,t:40,b:75},xaxis:{range:[0,110]}},
      note:'Descriptivo y sensible a tamaños muestrales reducidos. No implica que la doble postulación produzca la elección.'});
   const o=[
    {resultado:'Ninguna candidatura electa',casos:filtered.filter(r=>!r.electa_diputacion&&!r.electa_presidencia).length},
    {resultado:'Solo diputación electa',casos:filtered.filter(r=>r.electa_diputacion&&!r.electa_presidencia).length},
    {resultado:'Solo Presidencia electa',casos:filtered.filter(r=>r.electa_presidencia&&!r.electa_diputacion).length},
    {resultado:'Ambos cargos electos',casos:filtered.filter(r=>r.electa_diputacion&&r.electa_presidencia).length}
   ];
   plot('Resultados de las postulaciones dobles · ambos cargos',
    [{type:'bar',orientation:'h',x:o.map(r=>r.casos),y:o.map(r=>r.resultado),text:o.map(r=>`${r.casos} · ${fmt(100*r.casos/filtered.length,1)} %`),textposition:'outside',cliponaxis:false,marker:{color:['#95a6b1','#14857f','#c58351','#684c93']}}],o,
    {height:365,subtitle:`${scope}`,xTitle:'Casos persona–elección',layout:{margin:{l:205,r:80,t:40,b:75}},note:'Los resultados corresponden al mismo año de la doble postulación. Las proporciones tienen como denominador los casos del filtro activo; “electa” se toma de las bases de candidaturas y requiere contraste para discrepancias de fuente.'});
  }else note('No se observaron dobles postulaciones que cumplan esta combinación de filtros. Revise rango temporal, provincia o resultado.',true);
  const nRepeat=uniq(filtered.filter(r=>r.repeticion_doble).map(r=>r.id_persona)).length;
  const episodes=filtered.filter(r=>r.repeticion_doble).length;
  add(`<div class="clarity-box"><b>Qué significa reiteración:</b> una persona presenta una doble postulación en una elección y vuelve a presentar <b>ambas candidaturas simultáneamente</b> en otra elección posterior. En la selección activa hay <b>${fmt(episodes)} episodios de reiteración</b> correspondientes a <b>${fmt(nRepeat)} personas nominales</b>. Ejemplo: si alguien realiza dobles postulaciones en tres elecciones, aporta <b>tres casos al total</b>, de los cuales <b>dos son reiteraciones</b>. Los cambios de partido o provincia se calculan entre dos episodios de doble postulación, no entre cualquier candidatura de la trayectoria.</div>`);
  table(filtered.map(r=>({...r,resultado_presidencial:r.electa_presidencia?'Electa':'No electa',resultado_legislativo:r.electa_diputacion?'Electa':'No electa',reiteracion_de_doble_postulacion:r.repeticion_doble?'Sí · posterior':'No · primera'})),
   ['anio','nombre','sexo','partido_presidencial','partido_legislativo','provincia','orden_lista','resultado_presidencial','resultado_legislativo','reiteracion_de_doble_postulacion','anio_doble_anterior','continuidad_partido','continuidad_provincia','ambiguo'],
   'Auditoría nominal · dobles postulaciones simultáneas');
  add('<div class="clarity-box"><b>Sobre el posible beneficio electoral:</b> estos datos permiten calcular tasas descriptivas de éxito simultáneo; no permiten atribuir resultados a la doble postulación sin considerar posición de lista, magnitud provincial, voto del partido, selección de candidaturas y contexto político. Se distingue siempre una persona única de una persona–elección.</div>');
 }
 // V21.6 · Movimientos provinciales estrictamente legislativos.
 // No se usa población presidencial ni selector de población. Se valida que
 // AMBOS extremos del movimiento tengan una candidatura a diputación
 // documentada (misma identidad nominal, año y provincia).
 function movimientosLegislativos(){
   const raw=period(D.movimientos,false),scope=state.province;
   const incluir=(r,dim)=>scope==='Nacional'||(dim==='Provincia'?
     (r.origen===scope||r.destino===scope):true);
   // Se conserva el Sankey de cambios entre cargos, independiente del provincial.
   add('<div class="atlas17-subhead">Movimientos entre participaciones · cargos</div>');
   const cargos=raw.filter(r=>r.dimension==='Cargo'&&incluir(r,'Cargo'));
   pairedFlow(cargos,'Cargo',`Cambios de un cargo al siguiente · ${state.from}–${state.to}`);
   table(cargos,['nombre','anio_origen','origen','anio','destino','id_persona'],'Movimientos entre cargos · registros de origen y destino');
   add('<div class="atlas17-subhead">Movimientos entre participaciones · provincias</div>');
   const identificados=new Set((D.eventos||[])
     .filter(e=>e.cargo==='Diputación'&&P.includes(e.provincia))
     .map(e=>`${String(e.id_persona)}|${Number(e.anio)}|${e.provincia}`));
   const todos=raw.filter(r=>r.dimension==='Provincia'&&incluir(r,'Provincia'));
   const provincias=todos.filter(r=>
     identificados.has(`${String(r.id_persona)}|${Number(r.anio_origen)}|${r.origen}`)&&
     identificados.has(`${String(r.id_persona)}|${Number(r.anio)}|${r.destino}`));
   const nPeople=uniq(provincias.map(r=>r.id_persona)).length;
   add(`<div class="clarity-box"><b>Solo candidaturas legislativas.</b> `+
     `${num(provincias.length,0)} movimientos entre provincias, correspondientes a `+
     `${num(nPeople,0)} personas nominales en el intervalo seleccionado. `+
     `Cada transición exige una candidatura a diputación documentada en origen y destino. `+
     `${num(todos.length-provincias.length,0)} movimientos del registro general `+
     `no cumplen esa doble validación y quedan excluidos. `+
     `No representa migración residencial, doble postulación ni candidaturas presidenciales.</div>`);
   pairedFlow(provincias,'Provincia',`Diputaciones · siete provincias · ${state.from}–${state.to}`);
   if(provincias.length){
     const cells=[];for(const origen of P)for(const destino of P)
       cells.push({origen,destino,cambios:provincias.filter(r=>r.origen===origen&&r.destino===destino).length});
     plot('Matriz de movimientos provinciales · diputaciones',
       [{type:'heatmap',x:P,y:P,
         z:P.map(o=>P.map(d=>cells.find(r=>r.origen===o&&r.destino===d).cambios)),
         texttemplate:'%{z}',colorscale:[[0,'#edf4f6'],[1,'#167f83']],
         colorbar:{title:{text:'Cambios'}},
         hovertemplate:'Origen: %{y}<br>Destino: %{x}<br>%{z} movimientos<extra></extra>'}],
       cells,{height:450,subtitle:`Diputaciones · ${state.from}–${state.to}`,
         xTitle:'Provincia de destino',yTitle:'Provincia de origen',
         note:'Solamente movimientos entre candidaturas a diputación verificadas en ambas elecciones. La diagonal es cero: se cuentan cambios de provincia. No es una medición de migración residencial.'});
   }
   table(provincias,['nombre','sexo','anio_origen','origen','anio','destino','id_persona'],
     'Personas y movimientos provinciales · diputaciones');
   // Mantener la lectura partidaria opcional preexistente: no crear otras vistas provinciales.
   const optParty=local.movParty||'Ocultar';
   controls([['movParty','Lectura adicional de movimientos entre partidos',['Ocultar','Mostrar'],optParty]]);
   if(optParty==='Mostrar'){
     const parties=raw.filter(r=>r.dimension==='Partido'&&incluir(r,'Partido'));
     add('<div class="atlas17-subhead">Movimientos entre participaciones · partidos</div>');
     pairedFlow(parties,'Partido',`Cambios de denominación partidaria · ${state.from}–${state.to}`);
     table(parties,['nombre','anio_origen','origen','anio','destino','id_persona'],'Registros de movimientos partidarios');
   }
 }
 enhancedMobility=movimientosLegislativos;
 const beforeMobility=enhancedMobility;
 enhancedMobility=function(...args){const x=beforeMobility.apply(this,args);draw();return x;};
 window.ATLAS_V21_4_DOBLES_OK=true;
})();


'''

HTML_PATH = OUTPUT_DIR / 'Atlas_Electoral_Costa_Rica_1953_2026.html'
RECURSOS = PROJECT_DIR / 'recursos'
with TemporaryDirectory(prefix='atlas_v21_render_') as carpeta_temporal:
    temporal = Path(carpeta_temporal)
    for nombre in ('atlas_template.html','atlas_v3.js','atlas_final.js','atlas_v12.js','atlas_v14.js'):
        shutil.copyfile(RECURSOS / nombre, temporal / nombre)
    # La extensión se coloca al FINAL, después de las capas V12/V14.
    # Así no sobrescribe los recursos permanentes ni duplica visualizaciones.
    ultima = temporal / 'atlas_v14.js'
    ultima.write_text(ultima.read_text(encoding='utf-8')+'\n'+ATLAS_V21_AJUSTES_JS,encoding='utf-8')
    render_app(data, temporal / 'atlas_template.html', HTML_PATH)
(OUTPUT_DIR/'Atlas_Electoral_Figuras_Historicas.html').unlink(missing_ok=True)
print('Aplicación pública actualizada:', HTML_PATH.resolve())
print('Sankey: porcentajes visibles en ambos lados del nodo y lectura adicional por cursor.')
print('Género presidencial: hitos comparativos visibles en Presidencia y Vicepresidencias.')
print('Únicas salidas: HTML público y datos. No se crean figuras externas.')


Aplicación pública actualizada: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
Sankey: porcentajes visibles en ambos lados del nodo y lectura adicional por cursor.
Género presidencial: hitos comparativos visibles en Presidencia y Vicepresidencias.
Únicas salidas: HTML público y datos. No se crean figuras externas.


### 21.9 · Ajustes de presentación no destructivos

Se mejoran **exclusivamente** tipografía de controles, ayudas de lectura, estados de foco y distribución de encabezados en pantallas pequeñas. La gráfica Plotly, sus escalas, filtros, trazas, colores, cálculos, descarga, título y tamaños **no se modifican**. La hoja CSS es autosuficiente y se inserta una sola vez al final de la generación del HTML. Los recursos originales permanecen intactos.

In [33]:
# Atlas V21.9 · Refinamiento conservador de interfaz. CSS estático.
# Este bloque no modifica los recursos externos ni el contenido de las figuras.
from pathlib import Path
from tempfile import NamedTemporaryFile
import os

HTML_PUBLICO = OUTPUT_DIR / "Atlas_Electoral_Costa_Rica_1953_2026.html"
assert HTML_PUBLICO.is_file(), "Ejecutá primero la celda que genera el HTML público."
MARCA = "<!-- ATLAS_V21_9_INTERFAZ -->"
CSS_PUBLICO = r"""
<style id="atlas-v21-9-interfaz">
/* Ajustes de accesibilidad y jerarquía para elementos externos a Plotly. */
.navbutton:focus-visible, .button:focus-visible, .exports button:focus-visible,
.intro-button:focus-visible, .intro-link:focus-visible,
select:focus-visible, input:focus-visible, summary:focus-visible,
a:focus-visible {outline:3px solid #157e87;outline-offset:3px;box-shadow:0 0 0 3px #ffffff;}
.navbutton {transition:background-color .15s ease,color .15s ease;}
.section-head h2 {line-height:1.22;letter-spacing:-.55px;}
.section-head p {line-height:1.65;}
.scope {font-weight:650;letter-spacing:.12px;}
.controls {border:1px solid #dce7ea;row-gap:14px;}
.controls label {line-height:1.4;}
.controls select {min-width:125px;}
.card-head > div:first-child {min-width:0;}
.card-head h3 {overflow-wrap:break-word;line-height:1.38;}
.card-head .mini {line-height:1.55;display:block;}
.exports button:hover {border-color:#9dbdc7;background:#f0f6f7;color:#204858;}
.chart-note {line-height:1.65;border-top:1px solid #edf1f3;padding-top:10px;margin-top:12px;}
.reading-help > summary {line-height:1.5;}
.notice {line-height:1.62;}
.tablefooter {gap:12px;flex-wrap:wrap;}
@media(max-width:750px){
  .card-head {flex-wrap:wrap;row-gap:10px;}
  .card-head .exports {margin-left:auto;}
  .section-head {flex-wrap:wrap;}
  .controls {gap:12px;}
  .controls label {flex:1 1 170px;min-width:0;}
}
@media(prefers-reduced-motion:reduce){.navbutton {transition:none;}}
</style>
"""

# Idempotencia: no repetir la hoja ni crear archivos secundarios.
contenido = HTML_PUBLICO.read_text(encoding="utf-8")
if MARCA not in contenido:
    cierre = contenido.find("</head>")
    if cierre < 0:
        raise ValueError("HTML público sin </head>: no se aplica una transformación insegura.")
    contenido = contenido[:cierre] + MARCA + "\n" + CSS_PUBLICO + "\n" + contenido[cierre:]
    HTML_PUBLICO.write_text(contenido,encoding="utf-8")
    estado = "Insertada una vez"
else:
    estado = "Ya presente; sin duplicación"
assert HTML_PUBLICO.read_text(encoding="utf-8").count(MARCA)==1
print("Interfaz V21.9:",estado)
print("HTML público:",HTML_PUBLICO.resolve())
print("Sin nuevas carpetas, JS permanente ni figuras externas.")


Interfaz V21.9: Insertada una vez
HTML público: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
Sin nuevas carpetas, JS permanente ni figuras externas.


## 22.0 · Archivo presidencial, hemiciclo y recursos de consulta\nEsta actualización amplía módulos de consulta pública sin rediseñar las visualizaciones Plotly aprobadas. Los recursos de la interfaz son `recursos/atlas_v22_public.js` y `recursos/atlas_v22_public.css`; el último bloque integra ambos en el HTML autónomo.\n

In [34]:
# V22.0 · Mejoras públicas no destructivas, con recursos fuente auditables.
# Se ejecuta DESPUÉS de la generación estable V21.9. No modifica el JavaScript
# de gráficas aprobadas: conserva sus traces, cálculo, exportación, nombres y filtros.
from pathlib import Path

html_path = OUTPUT_DIR / 'Atlas_Electoral_Costa_Rica_1953_2026.html'
resource_dir = PROJECT_DIR / 'recursos'
assert html_path.is_file() and (resource_dir/'atlas_v22_public.js').is_file() and (resource_dir/'atlas_v22_public.css').is_file()
html = html_path.read_text(encoding='utf-8')
js = (resource_dir/'atlas_v22_public.js').read_text(encoding='utf-8')
css = (resource_dir/'atlas_v22_public.css').read_text(encoding='utf-8')
marker_js = '/* ATLAS_V22_PUBLIC_PRODUCT'
marker_css = '/* ATLAS_V22_PUBLIC_PRESENTATION'
js_hook = "window.addEventListener('resize',()=>document.querySelectorAll('.js-plotly-plot').forEach(el=>Plotly.Plots.resize(el)));setup();"
assert js_hook in html, 'No se encontró el punto de integración del HTML V21.9.'
# La celda es idempotente: si se reejecuta, NO incorpora una segunda copia.
if marker_js not in html:
    html = html.replace(js_hook, js + '\n' + js_hook, 1)
if marker_css not in html:
    assert '</head>' in html
    html = html.replace('</head>', '<style id="atlas-v22-interfaz">\n' + css + '\n</style>\n</head>', 1)
html_path.write_text(html,encoding='utf-8')
assert html.count(marker_js)==1 and html.count(marker_css)==1
print('Atlas V22 público:',html_path.resolve())
print('Módulos ampliados: fórmulas, Asamblea histórica, diccionario, laboratorio y metodología.')
print('Visualizaciones originales, Excel fuente, cálculos y exportaciones sin modificaciones.')


Atlas V22 público: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
Módulos ampliados: fórmulas, Asamblea histórica, diccionario, laboratorio y metodología.
Visualizaciones originales, Excel fuente, cálculos y exportaciones sin modificaciones.


### V22 · Mapas provinciales sincronizados (corrección de ubicación y lectura)

- **Sección 07 · Presidencia y Asamblea:** ubica los mapas de ganadores provinciales **inmediatamente después del gráfico de composición de la Asamblea**, no al final de todas las tablas. Muestra en paralelo resultados presidenciales de primera ronda y legislativos, con colores partidarios y detalle al pasar el cursor. Actualiza los dos mapas cuando se cambia el año, incluida la reproducción histórica.
- **Sección 18 · Mapas provinciales:** mantiene los valores de participación y la comparación con la elección anterior. Si el usuario solicita "Cambio desde la elección previa" en 1953, muestra el detalle observado y explica por qué no existe una diferencia histórica calculable. En años posteriores la variación se expresa en puntos porcentuales con colores divergentes según signo.

**Datos y módulos restantes preservados.** No modifica los siete archivos Excel ni las gráficas de otras secciones. Se puede reejecutar esta celda para reemplazar una versión antigua del mismo parche sin duplicarlo.

**Ajuste de encuadre V22:** ambos mapas provinciales se reajustan cuando terminan de dibujarse *y ya se encuentran dentro de su cuadrícula*, con escala geográfica uniforme y centrado continental. Se preservan los colores, las etiquetas, los datos y la exportación original.


In [35]:
# V22 · Mapas provinciales sincronizados y contraste histórico (extensión autónoma).
# No altera recursos fuente, siete Excel, gráficas previas ni crea figuras externas.
# Se inserta una sola vez dentro del HTML público generado por la celda V22.
from pathlib import Path
from tempfile import NamedTemporaryFile
import os

HTML_V22_MAPAS = OUTPUT_DIR / "Atlas_Electoral_Costa_Rica_1953_2026.html"
assert HTML_V22_MAPAS.is_file(), "Primero ejecute el generador HTML V22 del notebook."
V22_MAPS_JS = '/* ATLAS_V22_1_MAPAS_HISTORICOS_Y_PROVINCIALES | Mejora no destructiva. */\n(function(){\n \'use strict\';\n if(typeof views===\'undefined\'||typeof D===\'undefined\'||typeof P===\'undefined\'||typeof plot!==\'function\')return;\n const yearsAll=(typeof Y!==\'undefined\'?Y:(D.meta?.años||[])).map(Number).sort((a,b)=>a-b);\n const signedValue=x=>x!==null&&x!==undefined&&x!==\'\'&&Number.isFinite(Number(x));\n const numeric=x=>signedValue(x)?Number(x):null;\n const nearRows=(arr,province,year)=>arr.find(r=>r.provincia===province&&Number(r.anio)===year);\n // Sección 18: un recorte 2022–2026 no debe eliminar 2018 como referencia para 2022.\n if(typeof atlas14MapRows===\'function\'){\n  const originalMapRows=atlas14MapRows;\n  atlas14MapRows=function(mode,arena,party,key){\n   if(mode!==\'Cambio desde la elección previa\'&&mode!==\'Brecha presidencial–legislativa\')return originalMapRows(mode,arena,party,key);\n   const foco=Number(state.focus),idx=yearsAll.indexOf(foco);\n   const anterior=idx>0?yearsAll[idx-1]:null;\n   if(mode===\'Brecha presidencial–legislativa\'){\n    return P.map(provincia=>{\n     let p=null,l=null;\n     if(party===\'Todos\'){\n      p=(D.metricas||[]).find(r=>r.provincia===provincia&&Number(r.anio)===foco&&r.arena===\'Presidencial\'&&Number(r.ronda)===1)?.participacion_pct;\n      l=(D.metricas||[]).find(r=>r.provincia===provincia&&Number(r.anio)===foco&&r.arena===\'Legislativa\'&&Number(r.ronda)===1)?.participacion_pct;\n     }else{\n      const r=(D.panel||[]).find(r=>r.provincia===provincia&&Number(r.anio)===foco&&r.partido===party);\n      p=r?.porcentaje_Presidencial;l=r?.porcentaje_Legislativa;\n     }\n     return {anio:foco,provincia,valor:signedValue(p)&&signedValue(l)?numeric(p)-numeric(l):null,etiqueta:party===\'Todos\'?\'Participación presidencial − legislativa (pp)\':\'Voto presidencial − legislativo (pp)\'};\n    });\n   }\n   const base=party===\'Todos\'?(key===\'indice_representacion\'?(D.representacion_territorial||[]):(D.metricas||[]).filter(r=>r.arena===arena&&Number(r.ronda)===1)):(D.panel||[]).filter(r=>r.partido===party);\n   return P.map(provincia=>{\n    const actual=nearRows(base,provincia,foco)?.[key],previo=anterior===null?null:nearRows(base,provincia,anterior)?.[key];\n    return {anio:foco,provincia,anio_anterior:anterior,valor:signedValue(actual)&&signedValue(previo)?numeric(actual)-numeric(previo):null,\n      etiqueta:`Cambio de ${metricLabels[key]||key} · ${anterior??\'sin elección anterior\'} → ${foco}${/(_pct|porcentaje)/i.test(key)?\' (pp)\':\'\'}`};\n   });\n  };\n }\n // Paleta divergente idéntica en celdas y leyenda: magenta negativo, verde positivo.\n if(typeof atlas14MapFigure===\'function\'){\n  atlas14MapFigure=function(rows,titleText,opts={}){\n   const vals=rows.filter(r=>signedValue(r.valor)).map(r=>Number(r.valor)),lo=vals.length?Math.min(...vals):0,hi=vals.length?Math.max(...vals):0;\n   const divergent=!!opts.delta||lo<0,span=Math.max(1e-8,Math.abs(lo),Math.abs(hi));\n   const hexToRgb=h=>[1,3,5].map(i=>parseInt(h.slice(i,i+2),16));\n   const mix=(a,b,t)=>{const c=hexToRgb(a),d=hexToRgb(b);return \'rgb(\'+c.map((v,i)=>Math.round(v+(d[i]-v)*Math.max(0,Math.min(1,t)))).join(\',\')+\')\';};\n   const color=v=>{\n    if(!signedValue(v))return \'#e4e8eb\';\n    v=Number(v);\n    if(divergent)return v<0?mix(\'#f2f6f6\',\'#b64b78\',Math.abs(v)/span):mix(\'#f2f6f6\',\'#168572\',v/span);\n    return mix(\'#edf7f7\',\'#097c87\',(v-lo)/Math.max(hi-lo,1e-8));\n   };\n   const tr=[];\n   for(const f of D.geografia?.features||[]){\n    const name=f.properties.NAME_1,row=rows.find(r=>r.provincia===name),v=row?.valor;\n    const polygons=f.geometry.type===\'MultiPolygon\'?f.geometry.coordinates:[f.geometry.coordinates];\n    for(const poly of polygons){const ring=poly[0];if(!ring?.length)continue;\n     tr.push({type:\'scatter\',mode:\'lines\',x:ring.map(p=>p[0]),y:ring.map(p=>p[1]),fill:\'toself\',fillcolor:color(v),line:{color:\'#fff\',width:1},showlegend:false,hoveron:\'fills\',hoverinfo:\'text\',\n      text:`${name} · ${state.focus}<br>${row?.etiqueta||opts.label||\'Indicador\'}: ${signedValue(v)?num(v,2):\'Sin dato\'}`});\n    }\n   }\n   const barColors=divergent?[[0,\'#b64b78\'],[.5,\'#f2f6f6\'],[1,\'#168572\']]:[[0,\'#edf7f7\'],[1,\'#097c87\']];\n   tr.push({type:\'scatter\',mode:\'markers\',x:[null],y:[null],showlegend:false,marker:{size:0,color:[0],cmin:divergent?-span:lo,cmax:divergent?span:(hi===lo?lo+1:hi),colorscale:barColors,showscale:true,colorbar:{title:{text:opts.label||\'Valor\'},thickness:15}}});\n   plot(titleText,tr,rows,{height:590,subtitle:`${state.focus} · ${opts.subtitle||\'\'}`,\n    layout:{xaxis:{visible:false,range:[-86.2,-82.4]},yaxis:{visible:false,scaleanchor:\'x\',scaleratio:1,range:[8,11.35]},hovermode:\'closest\',showlegend:false,margin:{l:18,r:80,t:25,b:18}},\n    note:!vals.length?\'No existe un valor comparable para esta elección y selección. El primer proceso de la serie no tiene elección anterior.\':(opts.note||\'La misma escala cromática se aplica al mapa y a su leyenda. Consulte el valor exacto al pasar el cursor.\')});\n  };\n }\n // La primera elección no tiene comparación anterior. Si se solicita un cambio\n // para 1953, se ofrece el detalle real y una explicación visible, no un mapa vacío.\n if(typeof views.mapas===\'function\'){\n  const previousMapsView=views.mapas;\n  views.mapas=function(...args){\n   const first=Number(state.focus)===yearsAll[0] && local.mapmode===\'Cambio desde la elección previa\';\n   if(first)local.mapmode=\'Detalle provincial\';\n   const out=previousMapsView.apply(this,args);\n   if(first){\n    const area=document.getElementById(\'content\');\n    const msg=document.createElement(\'p\');msg.className=\'notice atlas-v22-primera-eleccion\';\n    msg.textContent=`${yearsAll[0]} es el primer año registrado: no existe una elección anterior con la cual calcular el cambio. Se muestra el valor provincial observado. Para comparar, seleccione ${yearsAll[1]} o una elección posterior.`;\n    const controls=area?.querySelector(\'.controls\');\n    if(controls)controls.insertAdjacentElement(\'afterend\',msg);\n    else area?.prepend(msg);\n   }\n   return out;\n  };\n }\n // Sección 07. Mapas del partido con mayor votación provincial, por arena, en primera ronda.\n const customColors={\n  \'NUEVA REPUBLICA\':\'#39bced\',\'PUEBLO SOBERANO\':\'#0da4b5\',\n  \'RESTAURACION NACIONAL\':\'#1d84d6\',\'UNIDAD SOCIAL CRISTIANA\':\'#0057b8\',\n  \'UNIDAD\':\'#6d7d88\',\'UNIFICACION NACIONAL\':\'#a7b2ba\',\n  \'REPUBLICANO\':\'#caa11a\',\'REPUBLICANO NACIONAL\':\'#caa11a\'\n };\n function partyColor(name){\n  const normalized=typeof norm===\'function\'?norm(name):String(name||\'\').normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').toUpperCase();\n  return customColors[normalized]||(typeof semanticColor===\'function\'?semanticColor(name):\'#748996\')||\'#748996\';\n }\n function winnersOfYear(arena,year){\n  const data=(D.metricas||[]).filter(r=>Number(r.anio)===year&&r.arena===arena&&Number(r.ronda)===1&&P.includes(r.provincia));\n  return P.map(p=>{\n   const r=data.find(v=>v.provincia===p),ganador=r?.ganador||null;\n   const vote=(D.votos||[]).find(v=>Number(v.anio)===year&&v.provincia===p&&v.arena===arena&&Number(v.ronda)===1&&v.partido===ganador);\n   return {anio:year,arena,provincia:p,partido:ganador||\'Sin dato\',porcentaje:numeric(r?.concentracion_1),votos:numeric(vote?.votos),fuente:\'Primera ronda / voto provincial\'};\n  });\n }\n function generateProvinceMap(arena,year){\n  const rows=winnersOfYear(arena,year),tr=[],seen=new Set();\n  const palette=[...new Set(rows.map(r=>r.partido))].filter(p=>p!==\'Sin dato\');\n  for(const f of D.geografia?.features||[]){\n   const name=f.properties.NAME_1,rr=rows.find(r=>r.provincia===name),p=rr?.partido||\'Sin dato\';\n   const color=p===\'Sin dato\'?\'#e4e8eb\':partyColor(p);\n   const polygons=f.geometry.type===\'MultiPolygon\'?f.geometry.coordinates:[f.geometry.coordinates];\n   for(const poly of polygons){const ring=poly[0];if(!ring?.length)continue;\n    const hover=`${name} · ${year}<br>Primera fuerza ${arena===\'Presidencial\'?\'presidencial\':\'legislativa\'}: ${p}`+\n     `<br>Votos válidos: ${rr?.votos===null?\'No disponible\':num(rr?.votos,0)}`+\n     `<br>Cuota de votos válidos: ${rr?.porcentaje===null?\'Sin dato\':num(rr.porcentaje,2)+\' %\'}`;\n    tr.push({type:\'scatter\',mode:\'lines\',x:ring.map(v=>v[0]),y:ring.map(v=>v[1]),fill:\'toself\',fillcolor:color,\n      line:{color:\'#fff\',width:1.35},showlegend:false,hoveron:\'fills\',hoverinfo:\'text\',text:hover});\n   }\n  }\n  const nombre=arena===\'Presidencial\'?\'Resultados presidenciales por provincia\':\'Resultados legislativos por provincia\';\n  const old=new Set(document.querySelectorAll(\'#content .card\'));\n  plot(nombre,tr,rows,{height:460,subtitle:`${year} · ${arena===\'Presidencial\'?\'Primera ronda\':\'Diputaciones\'} · siete provincias`,\n   layout:{xaxis:{visible:false,range:[-86.2,-82.4]},yaxis:{visible:false,scaleanchor:\'x\',scaleratio:1,range:[8,11.35]},margin:{l:5,r:5,t:10,b:15},showlegend:false,hovermode:\'closest\'},\n   note:arena===\'Presidencial\'?\'El color indica el partido más votado en primera ronda en cada provincia. No necesariamente corresponde al ganador final de la Presidencia si hubo segunda ronda.\':\'El color indica el partido con más votos para diputaciones en cada provincia; no implica que obtuvo todos los escaños provinciales.\'});\n  const newCard=[...document.querySelectorAll(\'#content .card\')].find(node=>!old.has(node));\n  if(newCard){newCard.classList.add(\'atlas-v22-provincia-card\');\n   const legend=document.createElement(\'div\');legend.className=\'atlas-v22-provincia-leyenda\';\n   for(const p of palette){const item=document.createElement(\'span\');item.className=\'atlas-v22-provincia-partido\';\n    const swatch=document.createElement(\'i\');swatch.style.backgroundColor=partyColor(p);item.append(swatch,document.createTextNode(typeof labelText===\'function\'?labelText(p):p));legend.append(item);}\n   newCard.appendChild(legend);\n  }\n  return newCard;\n }\n function applyFormulaColors(){\n  const section=[...document.querySelectorAll(\'#content .card\')].find(c=>/Fórmula presidencial electa/i.test(c.querySelector(\'h3\')?.textContent||\'\'));\n  if(!section)return;\n  const elected=(D.ejecutivo_electo||[]).filter(r=>Number(r.anio)===Number(state.focus));\n  section.querySelectorAll(\'.assembly-person\').forEach((element,i)=>{\n   const candidate=elected[i];if(!candidate)return;\n   element.style.borderLeft=\'4px solid \'+partyColor(candidate.partido);\n   const spans=[...element.querySelectorAll(\'span\')];\n   if(spans.length){const s=spans[spans.length-1];s.style.color=partyColor(candidate.partido);s.style.fontWeight=\'750\';}\n  });\n }\n let currentYear=null;\n function refreshYear(force){\n  if(state.tab!==\'gobierno\')return;\n  const year=Number(state.focus);\n  if(!yearsAll.includes(year))return;\n  const active=document.getElementById(\'atlas-v22-map-pair\');\n  if(active&&currentYear===year&&!force)return;\n  if(active)active.remove();\n  currentYear=year;\n  const host=document.createElement(\'section\');host.id=\'atlas-v22-map-pair\';\n  host.className=\'atlas-v22-provincia-wrap\';\n  host.innerHTML=`<div class="atlas-v22-provincia-head"><div><h3>Mapa electoral provincial · ${year}</h3><p>Primera fuerza por votos en cada provincia · recorrido histórico sincronizado con la Asamblea.</p></div></div><div class="atlas-v22-provincia-grid"></div>`;\n  document.querySelector(\'#content\')?.appendChild(host);\n  const grid=host.querySelector(\'.atlas-v22-provincia-grid\');\n  const left=generateProvinceMap(\'Presidencial\',year),right=generateProvinceMap(\'Legislativa\',year);\n  if(left)grid.appendChild(left);if(right)grid.appendChild(right);\n  // Después de mover ambas tarjetas del contenido general a una cuadrícula,\n  // Plotly conserva por un instante el ancho original de página completa.\n  // Esperar a que sus newPlot finalicen y reajustar en el contenedor real evita\n  // que el mapa se abra cortado, desplazado o pequeño en el PNG exportado.\n  const newMapPlots=typeof pending!==\'undefined\'?pending.slice(-2):[];\n  const centerProvincialMaps=async()=>{\n   if(!host.isConnected||state.tab!==\'gobierno\'||Number(state.focus)!==year)return;\n   const nodes=[...grid.querySelectorAll(\'.atlas-v22-provincia-card .js-plotly-plot\')];\n   for(const gd of nodes){\n    if(!gd.isConnected||!gd._fullLayout)continue;\n    await Plotly.Plots.resize(gd);\n    if(!gd.isConnected)continue;\n    // Límites continentales del GeoJSON: se preserva la proporción cartográfica.\n    // Isla del Coco no altera el encuadre de las siete provincias continentales.\n    const size=gd._fullLayout?._size||{};\n    const w=Math.max(150,Number(size.w)||gd.clientWidth-10);\n    const h=Math.max(150,Number(size.h)||gd.clientHeight-25);\n    const midLon=(-85.96069335937489-82.55322265625)/2;\n    const midLat=(8.037637710571346+11.21691799163824)/2;\n    const spanLon=85.96069335937489-82.55322265625;\n    const spanLat=11.21691799163824-8.037637710571346;\n    const unitsPerPixel=Math.max(spanLon*1.08/w,spanLat*1.08/h);\n    const dx=unitsPerPixel*w/2,dy=unitsPerPixel*h/2;\n    await Plotly.relayout(gd,{\n     \'xaxis.range\':[midLon-dx,midLon+dx],\n     \'yaxis.range\':[midLat-dy,midLat+dy],\n     \'xaxis.autorange\':false,\'yaxis.autorange\':false\n    });\n   }\n  };\n  Promise.allSettled(newMapPlots)\n   .then(()=>new Promise(resolve=>requestAnimationFrame(()=>requestAnimationFrame(resolve))))\n   .then(centerProvincialMaps)\n   .catch(err=>console.warn(\'Ajuste visual de mapas V22:\',err));\n  // Insertar inmediatamente después del hemiciclo, sin alterar el trazado original.\n  const hemi=[...document.querySelectorAll(\'#content > .card\')].find(e=>\n   /Composición partidaria de la Asamblea|Hemiciclo histórico|Hemiciclo/i.test(e.querySelector(\'h3\')?.textContent||\'\'));\n  if(hemi){hemi.insertAdjacentElement(\'afterend\',host);}\n  else{const formula=[...document.querySelectorAll(\'#content > .card\')].find(e=>/Fórmula presidencial electa/i.test(e.querySelector(\'h3\')?.textContent||\'\'));\n   if(formula)formula.insertAdjacentElement(\'afterend\',host);}\n  host.dataset.anio=String(year);\n  applyFormulaColors();\n }\n if(typeof views.gobierno===\'function\'){\n  const oldGov=views.gobierno;\n  views.gobierno=function(...args){const out=oldGov.apply(this,args);currentYear=null;refreshYear(true);return out;};\n }\n // Reproducir historia modifica el año: sincronizar SIN crear otro temporizador.\n const updateOnRender=typeof render===\'function\'?render:null;\n if(updateOnRender){render=async function(...args){const result=await updateOnRender.apply(this,args);if(state.tab===\'gobierno\')refreshYear(false);return result;};}\n const main=document.getElementById(\'content\');\n if(main&&typeof MutationObserver!==\'undefined\'){\n  const observer=new MutationObserver(()=>{\n   if(state.tab===\'gobierno\'&&yearsAll.includes(Number(state.focus))&&currentYear!==Number(state.focus))refreshYear(false);\n  });\n  observer.observe(main,{subtree:true,childList:true,characterData:true});\n }\n window.ATLAS_V22_1_MAPAS_OK=true;\n})();\n'
V22_MAPS_CSS = '/* ATLAS_V22_1_MAPAS_CSS: solo módulo de mapas históricos. */\n.atlas-v22-provincia-wrap{margin:22px 0 28px;}\n.atlas-v22-provincia-head{margin:0 0 12px;padding:1px 3px;}\n.atlas-v22-provincia-head h3{font-size:19px;line-height:1.35;letter-spacing:-.3px;color:#163e4d;}\n.atlas-v22-provincia-head p{font-size:12px;color:#526d7b;margin:3px 0 0;}\n.atlas-v22-provincia-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:16px;align-items:stretch;}\n.atlas-v22-provincia-card{min-width:0;box-sizing:border-box;margin:0;}\n.atlas-v22-provincia-card .card-head h3{font-size:15px;line-height:1.35;}\n.atlas-v22-provincia-card .chart{min-height:460px;}\n.atlas-v22-provincia-card .chart-note{line-height:1.52;}\n.atlas-v22-provincia-leyenda{margin:10px 0 0;display:flex;align-items:center;gap:8px 13px;flex-wrap:wrap;border-top:1px solid #e6edef;padding-top:12px;}\n.atlas-v22-provincia-partido{display:inline-flex;align-items:center;gap:6px;font-size:11px;line-height:1.3;color:#314d5b;}\n.atlas-v22-provincia-partido i{width:12px;height:12px;display:inline-block;border-radius:2px;flex-shrink:0;border:1px solid rgba(0,0,0,.1);}\n@media(max-width:1120px){.atlas-v22-provincia-grid{grid-template-columns:1fr;}.atlas-v22-provincia-card .chart{min-height:460px;}}\n'
V22_MAPS_JS_MARK = "/* ATLAS_V22_1_MAPAS_HISTORICOS_Y_PROVINCIALES"
V22_MAPS_CSS_MARK = "/* ATLAS_V22_1_MAPAS_CSS"
V22_HOOK = "window.addEventListener('resize',()=>document.querySelectorAll('.js-plotly-plot').forEach(el=>Plotly.Plots.resize(el)));setup();"

html = HTML_V22_MAPAS.read_text(encoding="utf-8")
if V22_MAPS_JS_MARK in html:
    # Refrescar la extensión antigua (V22.1) si esta celda se reejecuta aislada.
    start = html.index(V22_MAPS_JS_MARK)
    end = html.find("\n})();", start)
    assert end > start, "No se reconoció el final de la extensión V22 anterior."
    html = html[:start] + V22_MAPS_JS + html[end + len("\n})();"):]
else:
    assert V22_HOOK in html, "No se encontró el punto de integración de V22; no se ha modificado el HTML."
    html = html.replace(V22_HOOK,V22_MAPS_JS+"\n"+V22_HOOK,1)
if V22_MAPS_CSS_MARK not in html:
    assert "</head>" in html, "No se encontró la cabecera HTML."
    html = html.replace("</head>","<style id=\"atlas-v22-mapas-historicos\">\n"+V22_MAPS_CSS+"\n</style>\n</head>",1)
assert html.count(V22_MAPS_JS_MARK)==1 and html.count(V22_MAPS_CSS_MARK)==1
# Escritura atómica: no deja un HTML a medias en caso de error.
with NamedTemporaryFile("w",encoding="utf-8",dir=HTML_V22_MAPAS.parent,
                        prefix=".atlas_v22_mapas_",suffix=".html",delete=False) as temp:
    temp.write(html)
    temp_path=Path(temp.name)
os.replace(temp_path,HTML_V22_MAPAS)
print("V22 · dos mapas centrados y redimensionados en sus tarjetas, sincronizados con la historia.")
print("Mapas provinciales · cambio vs elección anterior y colores corregidos (solo sección 18).")
print("HTML generado:",HTML_V22_MAPAS.resolve())
print("No se generaron carpetas de figuras ni se modificaron archivos fuente.")


V22 · dos mapas centrados y redimensionados en sus tarjetas, sincronizados con la historia.
Mapas provinciales · cambio vs elección anterior y colores corregidos (solo sección 18).
HTML generado: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
No se generaron carpetas de figuras ni se modificaron archivos fuente.


**Comprobación editorial:** no se encontraron instrucciones personales de edición ni mensajes dirigidos al autor en la plantilla pública del Atlas. Las notas metodológicas que indican las limitaciones de los indicadores se conservan por razones de transparencia. La presentación V21.9 no cambia el orden de las secciones ni las interpretaciones; es una corrección visual acotada.

## Alcance de la versión 2.0

El recorrido cubre las dimensiones disponibles en las siete bases y preserva la agenda ampliada en `recursos/agenda.md`. Los análisis de ideología, parentesco, efectos causales, cronología jurídica exacta y cambios de bancada durante un mandato necesitan fuentes adicionales. Las trayectorias son nominales y la cobertura presidencial corresponde a las siete provincias de las bases. La aplicación permite explorar sin convertir cada asociación en un hallazgo académico.

En representación territorial, la serie longitudinal disponible permite comparar el **peso provincial en escaños** con el **peso provincial en el padrón electoral**. Esa comparación es descriptiva: no sustituye una medición basada en población ni la regla jurídica de asignación. V1.9 recupera la identificación visual de los años en que cambió la asignación provincial y documenta las provincias afectadas.


## Revisión 1.8 — claridad final de recurrencia, éxito y exploración

Esta revisión amplía **Éxito electoral** con dos lecturas separadas: (1) el universo completo, incluyendo personas nunca electas; y (2) el subconjunto de personas que sí resultaron electas, para observar con claridad cuántas ganaron en su primera elección documentada, cuántas necesitaron intentos previos y cuántas volvieron a resultar electas en años posteriores. También añade estadísticas explícitas de recurrencia, búsqueda avanzada del Explorador de personas, tarjetas auditables para antecedentes y una ampliación del Panorama histórico. Ninguna de estas mejoras modifica las siete bases fuente.

## Revisión 2.0 — versión final y pestaña de presentación

La versión 2.0 conserva íntegramente la arquitectura analítica de V1.9 y agrega una pestaña **00 · Presentación** como punto de entrada para el público. La pestaña explica qué es la aplicación, quién la desarrolló, el contexto IFED–TSE, el periodo y bases integradas, la metodología de Political Data Science, las principales dimensiones disponibles, cómo usar filtros y descargas y las cautelas necesarias para interpretar los resultados.

No se modifican las siete bases Excel fuente ni los cálculos de V1.9. El cambio funcional de V2.0 es de orientación, accesibilidad y presentación del producto final.


In [36]:
# Auditoría V1.9: representación territorial y cambios de asignación.
territorial = frames["representacion_territorial"].copy()
required_rep_cols = {
    "peso_escanos_pct", "peso_padron_pct", "brecha_representacion_pp",
    "indice_representacion", "electores_por_escano", "cambio_escanos",
    "balance_padron", "brecha_abs_pp", "cambio_asignacion"
}
assert required_rep_cols.issubset(territorial.columns)
assert territorial.groupby("anio").magnitud.sum().tolist() == [45, 45] + [57] * 17
cambios_asignacion = territorial[territorial.cambio_asignacion].copy()
assert cambios_asignacion.anio.nunique() >= 1
display(cambios_asignacion[["anio","provincia","magnitud","cambio_escanos"]].sort_values(["anio","provincia"]))


,anio,provincia,magnitud,cambio_escanos
14,1962,Alajuela,11.0,3.0
15,1962,Cartago,7.0,1.0
16,1962,Guanacaste,6.0,1.0
17,1962,Heredia,4.0,1.0
18,1962,Limón,3.0,1.0
19,1962,Puntarenas,6.0,1.0
20,1962,San José,20.0,4.0
21,1966,Alajuela,10.0,-1.0
24,1966,Heredia,3.0,-1.0
26,1966,Puntarenas,7.0,1.0


## Auditoría de doble postulación y movilidad provincial

La matriz aplica una **única escala cromática** a provincias por elección, totales históricos por provincia y totales nacionales por elección. **Los ceros son conteos válidos**. La casilla final del total país–periodo es la única que queda blanca y fuera de la escala. La movilidad provincial permanece restringida a candidaturas legislativas documentadas en ambos extremos.

**V21.8 (etiquetas)**: restaura los valores altos (2022: 10 en San José y 12 en Costa Rica; total histórico de San José: 33) mediante una capa de texto de alto contraste visible en el HTML y sus exportaciones, sin alterar la escala de color ni los conteos.
